# Previsão de Churn em Call Center — Áudio (AppTek) — versão Kaggle

Versão adaptada do notebook Databricks (mesmos passos, código e resultados): magics Databricks removidos e saídas de ficheiros para `./publicacao_kaggle/`.

**Como correr:** Internet **ON**, GPU OFF. O STEP 7 descarrega 160 WAVs (~2,2 GB) e extrai features acústicas — contar 30–60 min. Os ficheiros de saída (CSVs + PNGs) estão também disponíveis no dataset Kaggle associado a este notebook.

Pipeline: 873 conversas de call center (AppTek Callcenter Dialogues, CC BY-SA 4.0) → label proxy de risco de churn (texto do cliente, nunca feature) → features comportamentais + acústicas + do agente → Regressão Logística com split AGRUPADO POR FALANTE → F1 0,698 | AUC 0,708 no hold-out disjunto.

In [ ]:
# Compatibilidade Databricks → Kaggle/Jupyter
# No Databricks, `display()` vem incluído; no Kaggle/Jupyter não existe.
# Esta célula cria-o — o resto do notebook fica igual nas duas plataformas.
from IPython.display import display as _ipython_display

def display(x, **kwargs):
    """display() do Databricks adaptado: Spark DF → pandas → IPython display."""
    if hasattr(x, "toPandas"):
        x = x.toPandas()
    _ipython_display(x)


# Previsão de Churn em Call Center — Áudio de Clientes (AppTek)

Pipeline end-to-end de análise de churn a partir de **áudio real de call center** (dataset público AppTek — 873 conversas simuladas, 151h de áudio, 158 falantes, 14 acentos, 16 domínios — números verificados por execução), com transcrições, timestamps, metadados e género/role dos intervenientes.

**Origem do projeto:** o objetivo inicial era prever churn em português a partir de áudio real comercial, mas não havia ficheiros ou transcrições públicas — após pesquisa, validou-se o dataset AppTek em inglês, publicável (CC BY-SA 4.0), cobrindo múltiplos domínios e comportamento vocal.

**Objetivo:** prever, a partir de **features acústicas e comportamentais extraídas diretamente do áudio** (duração, ritmo, interrupções, energia, pitch), se o cliente indica **risco de churn**. O label é um proxy derivado da linguagem do cliente nas transcrições — união de churn explícito (cancelamento/fecho), frustração forte, escalonamento e ameaça de saída — documentado como limitação de pesquisa.

**Métrica de sucesso:** F1 da classe `churn` ≥ 0.70.

**Limitação:** as conversas são role-played (simulações por falantes reais, não chamados de produção); os labels de churn são proxies extraídos do texto do cliente — documenta-se como limitações de pesquisa.

**RESULTADO FINAL (encerrado 2026-11-15, todos os números verificados por execução):** modelo final — Regressão Logística (C=0,03) sobre 51 features comportamentais, acústicas e do texto do agente, avaliado em hold-out **disjunto por falante**: F1 churn 0,698 (meta 0,70 falhada por 0,002), AUC 0,708, precisão 83% quando sinaliza risco. Evolução 0,552 → 0,698 com split por falante + tuning (eliminou memorização de vozes). Uso recomendado: **ranker de priorização** com limiar p ≥ 0,50. 14 perguntas de negócio respondidas (células P1–P14). Artefactos de publicação (CSVs + README) exportados para `publicacao_kaggle/` — prontos para o Kaggle.

In [ ]:
%pip install -q librosa

In [ ]:
# ==============================================================================
# STEP 1: PREPARAÇÃO DO AMBIENTE E CARREGAMENTO DOS METADADOS E ÁUDIO (AppTek)
# ==============================================================================
# Objetivo: fixar configuração, descarregar metadados das 14 pastas de acento
# do AppTek Callcenter Dialogues (https://huggingface.co/datasets/apptek-com/apptek_callcenter_dialogues)
# para listar conversas e identificar ficheiros de áudio, transcrições e labels
# a derivar. Só metadados nesta célula — áudios e features extraímos em STEP 2/3.
#
# Input:  https://huggingface.co/datasets/apptek-com/apptek_callcenter_dialogues
# Output: df_meta — todas as conversas, segmentos, metadados, paths dos WAVs/JSONL
#
# Operações:
#   - Definir SEED, métrica de sucesso e dataset
#   - Descarga do metadados (JSONL) das 14 subpastas (cada acento inglês)
#   - Consolidação em dataframe único, visualização de estrutura
#   - Validação: domínios, falantes, roles, durações, paths de ficheiros
#   - Amostra de uma conversa, transcrição, segments e ficheiro WAV
# ===========================================================================

import json
import pandas as pd
import requests
import io
from pathlib import Path

# --- 1. Configuração do projeto -----------------------------------------------
SEED = 42
F1_MINIMO_ACEITAVEL = 0.70
DATASET_ID = "apptek-com/apptek_callcenter_dialogues"
ROOT_URL = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main"
API_TREE = f"https://huggingface.co/api/datasets/{DATASET_ID}/tree/main"

print(f"Métrica de sucesso: F1 da classe churn >= {F1_MINIMO_ACEITAVEL}")
print(f"Semente aleatória: {SEED}")

# --- Descobrir as pastas de acento da config 'diarization' (sem assumir nomes) ---
# Estrutura verificada: diarization/<acento>/metadata.jsonl (ex.: diarization/en-AU/)
r_tree = requests.get(f"{API_TREE}/diarization", timeout=60)
r_tree.raise_for_status()
pastas_acento = sorted(
    item["path"].rstrip("/").split("/")[-1]
    for item in r_tree.json()
    if item["type"] == "directory"
)
print(f"\nPastas de acento em diarization/ ({len(pastas_acento)}): {pastas_acento}")

# --- Descarregar o metadata.jsonl de cada acento e consolidar ------------------
metadados = []
for pasta in pastas_acento:
    resp = requests.get(f"{ROOT_URL}/diarization/{pasta}/metadata.jsonl", timeout=180)
    if resp.status_code == 200:
        linhas = resp.content.decode("utf-8").strip().splitlines()
        for linha in linhas:
            registo = json.loads(linha)
            registo["_pasta_acento"] = pasta          # origem do registo
            metadados.append(registo)
        print(f"  {pasta}: {len(linhas)} conversas")
    else:
        print(f"  {pasta}: sem metadata.jsonl (HTTP {resp.status_code})")

df_meta = pd.DataFrame(metadados)
print(f"\nTotal de conversas carregadas: {df_meta.shape[0]}")

# --- Estrutura real dos registos (chaves conforme o ficheiro, sem suposições) --
print("\nColunas de df_meta:", list(df_meta.columns))
primeiro = metadados[0]
print("Chaves do 1.º registo:", list(primeiro.keys()))
segs = primeiro.get("segments", [])
if segs:
    print(f"\n1.ª conversa: {len(segs)} segmentos")
    print("1.º segmento (300 chars):", json.dumps(segs[0], ensure_ascii=False)[:300])

print("\nSTEP 1 concluído: metadados das conversas carregados e estrutura real identificada.")

In [ ]:
# ==============================================================================
# STEP 2: EDA — PERFIL ESTATÍSTICO DOS METADADOS DAS CONVERSAS DE ÁUDIO
# ==============================================================================
# Objetivo: amostra exploratória dos metadados das 873 conversas — durações,
# domínios, acentos, segmentos, roles (agente vs cliente), falantes e géneros —
# com verificação final contra os números declarados no README do dataset.
#
# Input:  df_meta (STEP 1) — 873 conversas × [file_name, duration, domain,
#         accent, segments, _pasta_acento]
# Output: diagnósticos em consola (leitura apenas; df_meta não é alterado)
# ==============================================================================

import pandas as pd

# --- 1. Duração das conversas --------------------------------------------------
dur = df_meta["duration"]
print("=== Duração das conversas (minutos) ===")
print((dur / 60).describe().round(2).to_string())
print(f"\nÁudio total: {dur.sum()/3600:.1f} h | média: {dur.mean()/60:.1f} min | mediana: {dur.median()/60:.1f} min")

# --- 2. Domínios e acentos -----------------------------------------------------
print("\n=== Domínios (n.º de conversas) ===")
for k, v in df_meta["domain"].value_counts().items():
    print(f"  {k:22s} {v:4d}  ({v/len(df_meta):5.1%})")

print("\n=== Acentos (n.º de conversas) ===")
for k, v in df_meta["accent"].value_counts().sort_index().items():
    print(f"  {k:16s} {v:4d}")

# --- 3. Segmentos e roles (agente vs cliente) ----------------------------------
todos_segmentos = [s for segs in df_meta["segments"] for s in segs]
df_seg = pd.DataFrame(todos_segmentos)
print(f"\n=== Segmentos: {len(df_seg):,} no total ===")
print(df_seg["role"].value_counts().to_string())
print("\nGénero por role:")
print(df_seg.groupby(["role", "gender"]).size().to_string())

# --- 4. Falantes únicos --------------------------------------------------------
falantes = df_seg.drop_duplicates("speaker_id")[["speaker_id", "gender"]]
print(f"\nFalantes únicos: {len(falantes)} | géneros: {falantes['gender'].value_counts().to_dict()}")
print("Exemplos de speaker_id:", falantes["speaker_id"].head(3).tolist())

# --- 5. Verificação contra os números declarados no README ---------------------
esperado = {"conversas": 873, "segmentos": 94679, "falantes": 156, "dominios": 16, "horas": 151.1}
obtido = {
    "conversas": len(df_meta),
    "segmentos": len(df_seg),
    "falantes": len(falantes),
    "dominios": df_meta["domain"].nunique(),
    "horas": round(dur.sum() / 3600, 1),
}
print("\n=== Verificação vs README ===")
for k in esperado:
    estado = "OK" if obtido[k] == esperado[k] else "DIFERENTE"
    print(f"  {k:10s} esperado={esperado[k]:>8} obtido={obtido[k]:>8}  [{estado}]")

# --- 6. Amostra de uma conversa (primeiros 5 segmentos) -------------------------
ex = df_meta.iloc[0]
print(f"\n=== Amostra: {ex['file_name']} ({ex['domain']}, {ex['accent']}, {ex['duration']/60:.1f} min) ===")
for s in ex["segments"][:5]:
    print(f"  [{s['start']:6.1f}-{s['end']:6.1f}s] {s['role']:8s} ({s['gender']:6s}): {s['text'][:70]}")

print("\nSTEP 2 concluído: perfil dos metadados verificado — pronto para a limpeza (STEP 3).")

In [ ]:
# ==============================================================================
# STEP 3: LIMPEZA, INTEGRIDADE E NORMALIZAÇÃO EM TABELAS TIDY
# ==============================================================================
# Objetivo: validar a integridade dos metadados (duplicados, nulos, roles,
# segmentos inválidos) e normalizar os dados aninhados em duas tabelas tidy:
#   df_conversas  — 1 linha por conversa (873), com agregados comportamentais
#   df_segmentos  — 1 linha por segmento (94.679), nível falante/turno
#
# Input:  df_meta (STEP 1)
# Output: df_conversas e df_segmentos normalizadas, prontas para EDA de texto,
#         criação do label de churn e extração de features acústicas (WAVs)
#
# Operações:
#   - Checagens de integridade: duplicados, nulos, roles, durações <= 0
#   - Explosão dos segmentos em df_segmentos (com id_conversa herdado)
#   - Agregação por conversa: tempo de fala cliente/agente, nº palavras,
#     sobreposições (interrupções), textos consolidados por role
#   - Consistência: fim do último segmento vs duração declarada
# ==============================================================================

import pandas as pd

# --- 1. Checagens de integridade ---------------------------------------------
print("=== Integridade ===")
print(f"Conversas duplicadas (file_name): {df_meta['file_name'].duplicated().sum()}")
print(f"Registos com duration nulo: {df_meta['duration'].isna().sum()}")
print(f"Registos com domain/accent nulo: {df_meta[['domain', 'accent']].isna().sum().sum()}")

# Segmentos inválidos (fim <= início) e textos vazios, por conversa
seg_invalidos = 0
seg_texto_vazio = 0
for segs in df_meta["segments"]:
    for s in segs:
        if s["end"] <= s["start"]:
            seg_invalidos += 1
        if not str(s["text"]).strip():
            seg_texto_vazio += 1
print(f"Segmentos com fim <= início: {seg_invalidos}")
print(f"Segmentos com texto vazio: {seg_texto_vazio}")

roles = {s["role"] for segs in df_meta["segments"] for s in segs}
print(f"Roles presentes: {sorted(roles)}")

# --- 2. Explosão dos segmentos em tabela tidy (df_segmentos) ------------------
registros = []
for _, conv in df_meta.iterrows():
    for s in conv["segments"]:
        registros.append({
            "id_conversa": conv["file_name"].removesuffix(".wav"),
            "ficheiro_wav": conv["file_name"],
            "acento": conv["accent"],
            "dominio": conv["domain"],
            "inicio_s": s["start"],
            "fim_s": s["end"],
            "dur_seg_s": s["end"] - s["start"],
            "role": s["role"],
            "speaker_id": s["speaker_id"],
            "genero": s["gender"],
            "texto": str(s["text"]).strip(),
        })
df_segmentos = pd.DataFrame(registros)
print(f"\ndf_segmentos: {df_segmentos.shape[0]:,} linhas × {df_segmentos.shape[1]} colunas")

# --- 3. Agregados por conversa -----------------------------------------------
def contar_sobreposicoes(segs):
    """N.º de interrupções: turnos que começam antes de o anterior terminar."""
    ordenados = sorted(segs, key=lambda s: s["start"])
    n = 0
    for a, b in zip(ordenados, ordenados[1:]):
        if b["start"] < a["end"] and b["role"] != a["role"]:
            n += 1
    return n

por_conversa = df_segmentos.groupby("id_conversa", sort=False)
dur_por_role = por_conversa.apply(
    lambda d: pd.Series({
        "dur_fala_agente_s": d.loc[d["role"] == "agent", "dur_seg_s"].sum(),
        "dur_fala_cliente_s": d.loc[d["role"] == "customer", "dur_seg_s"].sum(),
    })
)
texto_por_role = por_conversa.apply(
    lambda d: pd.Series({
        "texto_agente": " ".join(d.loc[d["role"] == "agent", "texto"]),
        "texto_cliente": " ".join(d.loc[d["role"] == "customer", "texto"]),
    })
)

df_conversas = pd.DataFrame({
    "id_conversa": df_meta["file_name"].str.removesuffix(".wav").values,
    "ficheiro_wav": df_meta["file_name"].values,
    "acento": df_meta["accent"].values,
    "dominio": df_meta["domain"].values,
    "duracao_s": df_meta["duration"].values,
    "n_segmentos": df_meta["segments"].apply(len).values,
    "n_falantes": df_meta["segments"].apply(lambda segs: len({s["speaker_id"] for s in segs})).values,
    "n_sobreposicoes": df_meta["segments"].apply(contar_sobreposicoes).values,
}).set_index("id_conversa")
df_conversas = df_conversas.join(dur_por_role).join(texto_por_role)
df_conversas["pct_fala_cliente"] = (
    df_conversas["dur_fala_cliente_s"]
    / (df_conversas["dur_fala_cliente_s"] + df_conversas["dur_fala_agente_s"])
)
df_conversas["n_palavras_cliente"] = df_conversas["texto_cliente"].str.split().str.len()
df_conversas["n_palavras_agente"] = df_conversas["texto_agente"].str.split().str.len()
df_conversas = df_conversas.reset_index()

# --- 4. Consistência: fim do último segmento vs duração declarada -------------
fim_ultimo = df_segmentos.groupby("id_conversa")["fim_s"].max()
dif = (df_conversas.set_index("id_conversa")["duracao_s"] - fim_ultimo).abs()
print(f"\nDiferença média |duração declarada - fim do último segmento|: {dif.mean():.1f} s (max {dif.max():.1f} s)")

print(f"\ndf_conversas: {df_conversas.shape[0]} linhas × {df_conversas.shape[1]} colunas")
print("Colunas:", list(df_conversas.columns))
print(df_conversas[["duracao_s", "n_segmentos", "n_falantes", "n_sobreposicoes",
                    "pct_fala_cliente", "n_palavras_cliente"]].describe().round(2).to_string())

print("\nSTEP 3 concluído: tabelas tidy df_conversas e df_segmentos prontas (label de churn no STEP seguinte).")

In [ ]:
# ==============================================================================
# STEP 4: VISUALIZAÇÃO GERAL — TODAS AS VARIÁVEIS E PRIMEIRAS 77 LINHAS
# ==============================================================================
# Objetivo: inspeção visual da tabela limpa df_conversas — listagem completa
# das variáveis (nome, tipo, completude) e renderização das primeiras 77
# conversas para validação qualitativa dos dados normalizados.
#
# Input:  df_conversas (STEP 3) — 873 conversas × 15 colunas
# Output: tabela renderizada (display) das 77 primeiras linhas
#
# Nota: os textos consolidados são truncados a 70 caracteres APENAS na
# visualização — os dados originais em df_conversas ficam intactos.
# ==============================================================================

import pandas as pd

pd.set_option("display.max_columns", None)

print(f"Dimensões de df_conversas: {df_conversas.shape[0]} conversas × {df_conversas.shape[1]} variáveis\n")

# --- 1. Todas as variáveis: nome, tipo e completude ---------------------------
print("=== Todas as variáveis (nome | tipo | nulos) ===")
for col in df_conversas.columns:
    print(f"  {col:20s} {str(df_conversas[col].dtype):8s} nulos={df_conversas[col].isna().sum()}")

# --- 2. Primeiras 77 linhas (visualização) ------------------------------------
df_77 = df_conversas.head(77).copy()
df_77["texto_agente"] = df_77["texto_agente"].str.slice(0, 70) + "…"
df_77["texto_cliente"] = df_77["texto_cliente"].str.slice(0, 70) + "…"
display(df_77)

In [ ]:
# ==============================================================================
# STEP 5: CRIAÇÃO DO LABEL DE RISCO DE CHURN (PROXY) — LINGUAGEM DO CLIENTE
# ==============================================================================
# Objetivo: derivar churn_label (1 = RISCO de churn; 0 = caso contrário) a
# partir do TEXTO DO CLIENTE, com padrões explícitos e rastreáveis, e validar
# equilíbrio de classes e qualidade qualitativa dos matches antes de modelar.
#
# Input:  df_conversas (STEP 3) — coluna texto_cliente
# Output: df_conversas enriquecida com:
#   - churn_label     (int) — risco de churn = união A|B|C|D
#   - churn_explicito (int) — subconjunto A (cancelamento/fecho explícito)
#   - padroes_churn   (str) — componentes que dispararam (rastreabilidade)
#
# AVISO METODOLÓGICO (fuga de dados): o label DERIVA do texto do cliente —
# o texto do cliente NÃO pode entrar como feature (TF-IDF) na modelação.
# A previsão far-se-á com features acústicas, comportamentais e do texto do
# AGENTE (não usados na definição do label).
#
# História da definição: a v1 estrita (só churn explícito) deu 11/873 (1,3%)
# — insuficiente para treinar/testar. Por decisão do utilizador, alargou-se
# para RISCO DE CHURN (união de 4 componentes):
#   A. churn explícito — cancelar/fechar/terminar conta/serviço/subscrição/
#      plano/policy/contrato/membership; mudar de fornecedor; não renovar
#   B. frustração forte — 'fed up', 'ridiculous', 'unacceptable', 'worst',
#      'terrible', 'awful', 'furious', 'appalled', 'frustrated', ...
#   C. escalonamento — pedir gestor/supervisor, ameaçar queixa, ação legal
#   D. ameaça de saída — 'take my business elsewhere', 'switching banks',
#      'never using you again', ...
# 'Refund'/'money back' é CONTABILIZADO mas fica FORA da união (pedir
# reembolso não implica risco de churn).
# ==============================================================================

import re
import pandas as pd
from collections import Counter

# --- A. Churn explícito (v1, mantida na íntegra) -------------------------------
OBJETOS = r"(?:account|service|subscription|plan|policy|contract|membership)"
JANELA = r"[\w\s,'()-]{0,40}"   # tolera disfluências '(uh)' entre verbo e objeto

PADROES_EXPLICITO = {
    "cancelar": rf"\bcancel(?:l?ed|l?ing|lation)?\b{JANELA}\b{OBJETOS}\b",
    "fechar": rf"\bclose(?:d|ing)?\b{JANELA}\b{OBJETOS}\b",
    "terminar": rf"\bterminat(?:e|ed|ing|ion)\b{JANELA}\b{OBJETOS}\b",
    "mudar_de_fornecedor": r"\bswitch(?:ing)?\s+(?:to|over to)\s+(?:a\s+different|another|other)\s+(?:provider|company|bank|airline|carrier|supplier|service)\b",
    "nao_renovar": r"\b(?:not\s+renew(?:ing)?|won'?t\s+(?:be\s+)?renew(?:ing)?)\b",
}

# --- B. Frustração forte -------------------------------------------------------
PADRAO_FRUSTRACAO = (
    r"\b(?:fed\s+up|sick\s+and\s+tired|sick\s+of|ridiculous|unacceptable|"
    r"worst|terrible|awful|disgusting|disgusted|appalled|furious|"
    r"frustrated|frustrating|disappointed|disappointing|unhappy|not\s+happy|"
    r"dissatisfied|poor\s+service|bad\s+service|what\s+a\s+joke)\b"
)

# --- C. Escalonamento ---------------------------------------------------------
PADRAO_ESCALONAMENTO = (
    r"\b(?:speak\s+(?:to|with)\s+(?:a\s+|the\s+)?(?:manager|supervisor)|"
    r"talk\s+to\s+(?:a\s+|the\s+)?(?:manager|supervisor)|"
    r"your\s+(?:manager|supervisor)|"
    r"file\s+(?:a\s+)?complaint|make\s+(?:a\s+)?complaint|"
    r"report\s+(?:this|that|it)|take\s+this\s+further|"
    r"ombudsman|legal\s+action|lawyer|sue\s+you)\b"
)

# --- D. Ameaça de saída -------------------------------------------------------
PADRAO_SAIDA = (
    r"\b(?:take\s+my\s+business\s+elsewhere|"
    r"go(?:ing)?\s+(?:to|over\s+to)\s+(?:another|a\s+different)\s+"
    r"(?:provider|company|bank|airline|carrier|supplier|competitor)|"
    r"moving\s+to\s+(?:another|a\s+different)\s+(?:provider|company|bank|airline|carrier)|"
    r"switch(?:ing)?\s+(?:banks?|providers?|carriers?|companies)|"
    r"never\s+(?:using|coming)\s+(?:you\s+)?again|"
    r"done\s+with\s+(?:you|this))\b"
)

# Contabilizado mas FORA da união
PADRAO_REEMBOLSO = r"\b(?:refund|money\s+back)\b"

texto = df_conversas["texto_cliente"].astype(str)

# --- Aplicação dos componentes ------------------------------------------------
comp_expl = {nome: texto.str.contains(p, case=False, na=False, regex=True)
             for nome, p in PADROES_EXPLICITO.items()}
comp_frustracao = texto.str.contains(PADRAO_FRUSTRACAO, case=False, na=False, regex=True)
comp_escal = texto.str.contains(PADRAO_ESCALONAMENTO, case=False, na=False, regex=True)
comp_saida = texto.str.contains(PADRAO_SAIDA, case=False, na=False, regex=True)
comp_reembolso = texto.str.contains(PADRAO_REEMBOLSO, case=False, na=False, regex=True)

explicito = pd.DataFrame(comp_expl).any(axis=1)

print("=== Contribuição de cada componente (conversas) ===")
for nome, serie in comp_expl.items():
    print(f"  A. {nome:20s} {int(serie.sum()):4d}")
print(f"  B. frustracao          {int(comp_frustracao.sum()):4d}")
print(f"  C. escalonamento       {int(comp_escal.sum()):4d}")
print(f"  D. ameaca_saida        {int(comp_saida.sum()):4d}")
print(f"  -  reembolso           {int(comp_reembolso.sum()):4d}  [fora da união]")

df_conversas["churn_explicito"] = explicito.astype(int)
df_conversas["churn_label"] = (
    explicito | comp_frustracao | comp_escal | comp_saida
).astype(int)

# Rastreabilidade: componentes que dispararam em cada conversa
serie_grupo = {"frustracao": comp_frustracao, "escalonamento": comp_escal, "ameaca_saida": comp_saida}
df_conversas["padroes_churn"] = [
    ", ".join(
        [n for n, s in comp_expl.items() if s.iloc[i]]
        + [n for n, s in serie_grupo.items() if s.iloc[i]]
    )
    for i in range(len(df_conversas))
]

# --- 1. Equilíbrio de classes --------------------------------------------------
print("\n=== Equilíbrio de classes (risco de churn) ===")
taxa = df_conversas["churn_label"].mean()
for k, v in df_conversas["churn_label"].value_counts().sort_index().items():
    etiqueta = "não-churn" if k == 0 else "churn"
    print(f"  {etiqueta:10s} {v:4d}  ({v/len(df_conversas):.1%})")
if taxa < 0.05 or taxa > 0.60:
    print(f"AVISO: taxa {taxa:.1%} fora da faixa 5%–60% — discutir ajuste da definição.")
else:
    print(f"Taxa {taxa:.1%} — dentro da faixa aceitável para modelação.")

# Combinações de componentes nas conversas com churn (perfil do label)
grupos = []
for i in range(len(df_conversas)):
    g = set()
    if explicito.iloc[i]: g.add("A")
    if comp_frustracao.iloc[i]: g.add("B")
    if comp_escal.iloc[i]: g.add("C")
    if comp_saida.iloc[i]: g.add("D")
    grupos.append("+".join(sorted(g)) or "—")
perfil = Counter(g for g, lbl in zip(grupos, df_conversas["churn_label"]) if lbl == 1)
print("\nCombinações de componentes nas conversas churn:", dict(perfil.most_common()))

# --- 2. Taxa de churn por domínio ----------------------------------------------
print("\n=== Taxa de risco de churn por domínio ===")
td = df_conversas.groupby("dominio")["churn_label"].agg(["mean", "count"]).sort_values("mean", ascending=False)
for dom, linha in td.iterrows():
    print(f"  {dom:18s} {linha['mean']:6.1%}  (n={int(linha['count'])})")

# --- 3. Exemplos qualitativos: 1 excerto por componente ------------------------
def excerto(txt, padrao, contexto=60):
    m = padrao.search(txt)
    if m is None:
        return None
    return txt[max(0, m.start() - contexto):m.end() + contexto].replace("\n", " ")

print("\n=== Exemplos qualitativos (frase que disparou, por componente) ===")
exemplos_componentes = [
    ("A. churn explícito", explicito, re.compile("|".join(PADROES_EXPLICITO.values()), re.IGNORECASE)),
    ("B. frustração", comp_frustracao, re.compile(PADRAO_FRUSTRACAO, re.IGNORECASE)),
    ("C. escalonamento", comp_escal, re.compile(PADRAO_ESCALONAMENTO, re.IGNORECASE)),
    ("D. ameaça de saída", comp_saida, re.compile(PADRAO_SAIDA, re.IGNORECASE)),
]
for nome, serie, padrao in exemplos_componentes:
    for i, txt in enumerate(texto):
        if serie.iloc[i]:
            ex = excerto(txt, padrao)
            if ex:
                print(f"  {nome} [{df_conversas['id_conversa'].iloc[i]}]")
                print(f"    …{ex}…")
                break

# --- 4. Controlo negativo (2 conversas SEM label, 150 chars) -------------------
print("\n=== Controlo negativo: 2 conversas sem label ===")
sem = texto[df_conversas["churn_label"] == 0].sample(2, random_state=SEED)
for t in sem:
    print(" -", t[:150].replace("\n", " "))

print(f"\nSTEP 5 concluído: churn_label (risco) = {int(df_conversas['churn_label'].sum())} conversas; "
      f"churn_explicito = {int(df_conversas['churn_explicito'].sum())}.")

In [ ]:
# ==============================================================================
# STEP 6: FEATURES COMPORTAMENTAIS E DO AGENTE (873 conversas, sem fuga de dados)
# ==============================================================================
# Objetivo: construir o conjunto base de features PREDITORAS a partir dos
# metadados e segmentos — NÃO do conteúdo textual do cliente (que define o
# label). As features acústicas dos WAVs ficam para o STEP 7 (subset, por
# causa do peso do download).
#
# Input:  df_conversas e df_segmentos (STEP 3), churn_label (STEP 5)
# Output: df_features — 873 conversas × features numéricas + categóricas
#
# Features:
#   Comportamentais (dos timestamps e contagens):
#     - turnos_cliente / turnos_agente, densidade de turnos
#     - duração média de turno (cliente e agente)
#     - interrupções iniciadas pelo CLIENTE vs pelo AGENTE
#     - latência média de resposta do cliente (depois do agente)
#     - % de fala do cliente, % de silêncio (união de intervalos)
#     - ritmo de fala do cliente (palavras/s) — contagem, não conteúdo
#   Do TEXTO DO AGENTE (permitido — o label não deriva dele):
#     - pedidos de desculpa, agradecimentos, frases empáticas
#     - palavras por turno do agente, rácio agente/cliente de palavras
#   Categóricas: acento, domínio, género do cliente, género do agente
#
# Verificação de fuga de dados: nenhuma feature lê o CONTEÚDO do texto do
# cliente — apenas contagens de palavras e durações (comportamento vocal).
# ==============================================================================

import re
import numpy as np
import pandas as pd

# --- 1. Agregados por conversa a partir dos segmentos ordenados -------------
def tempo_fala_uniao(d):
    """Total de tempo com fala (união dos intervalos, sem dupla contagem)."""
    total = 0.0
    fim_cor = None
    for ini, fim in sorted(zip(d["inicio_s"], d["fim_s"])):
        if fim_cor is None or ini > fim_cor:
            total += fim - ini
            fim_cor = fim
        elif fim > fim_cor:
            total += fim - fim_cor
            fim_cor = fim
    return total

registros_feat = []
for idc, d in df_segmentos.groupby("id_conversa", sort=False):
    d = d.sort_values("inicio_s")
    turnos_c = int((d["role"] == "customer").sum())
    turnos_a = int((d["role"] == "agent").sum())
    interr_c = interr_a = 0
    latencias_c = []
    prev = None
    for row in d.itertuples():
        if prev is not None:
            if row.inicio_s < prev.fim_s and row.role != prev.role:
                if row.role == "customer":
                    interr_c += 1
                else:
                    interr_a += 1
            if row.role == "customer" and prev.role == "agent":
                gap = row.inicio_s - prev.fim_s
                if gap >= 0:
                    latencias_c.append(gap)
        prev = row
    tempo_fala = tempo_fala_uniao(d)
    registros_feat.append({
        "id_conversa": idc,
        "turnos_cliente": turnos_c,
        "turnos_agente": turnos_a,
        "interrupcoes_cliente": interr_c,
        "interrupcoes_agente": interr_a,
        "latencia_media_cliente_s": float(np.mean(latencias_c)) if latencias_c else np.nan,
        "tempo_fala_s": tempo_fala,
        "dur_media_turno_cliente_s": d.loc[d["role"] == "customer", "dur_seg_s"].mean(),
        "dur_media_turno_agente_s": d.loc[d["role"] == "agent", "dur_seg_s"].mean(),
        "genero_cliente": d.loc[d["role"] == "customer", "genero"].iloc[0],
        "genero_agente": d.loc[d["role"] == "agent", "genero"].iloc[0],
    })
df_feat_seg = pd.DataFrame(registros_feat)

# Verificação de consistência: interrupções cliente+agente = n_sobreposicoes (STEP 3)
soma_interr = df_feat_seg["interrupcoes_cliente"] + df_feat_seg["interrupcoes_agente"]
comp = soma_interr.values == df_conversas["n_sobreposicoes"].values
print(f"Consistência interrupções (cliente+agente == n_sobreposicoes): {int(comp.sum())}/{len(comp)} conversas OK")

# --- 2. Features derivadas das durações e contagens ---------------------------
base = df_conversas[["id_conversa", "acento", "dominio", "duracao_s", "n_segmentos",
                     "n_sobreposicoes", "dur_fala_agente_s", "dur_fala_cliente_s",
                     "pct_fala_cliente", "n_palavras_cliente", "n_palavras_agente",
                     "churn_label"]].copy()
base = base.merge(df_feat_seg, on="id_conversa", how="left", validate="one_to_one")

base["pct_silencio"] = 1 - base["tempo_fala_s"] / base["duracao_s"]
base["densidade_turnos"] = base["n_segmentos"] / (base["duracao_s"] / 60)
base["ritmo_palavras_cliente"] = base["n_palavras_cliente"] / base["dur_fala_cliente_s"]
base["ratio_palavras_agente_cliente"] = base["n_palavras_agente"] / base["n_palavras_cliente"].clip(lower=1)

# Latências ausentes (conversas sem transição agente→cliente medida) → mediana
n_lat_null = int(base["latencia_media_cliente_s"].isna().sum())
if n_lat_null:
    mediana_lat = base["latencia_media_cliente_s"].median()
    base["latencia_media_cliente_s"] = base["latencia_media_cliente_s"].fillna(mediana_lat)
    print(f"Latências ausentes preenchidas com a mediana ({mediana_lat:.2f} s): {n_lat_null} conversas")

# --- 3. Features do TEXTO DO AGENTE (permitido; label não deriva dele) --------
agente = df_conversas["texto_agente"].astype(str)
base["ag_pedidos_desculpa"] = agente.str.count(r"\b(?:sorry|apolog(?:y|ies|ize|se|zing))\b", flags=re.IGNORECASE)
base["ag_agradecimentos"] = agente.str.count(r"\b(?:thank\s+you|thanks)\b", flags=re.IGNORECASE)
base["ag_empaticas"] = agente.str.count(r"\b(?:i\s+understand|of\s+course|no\s+problem|absolutely|definitely|happy\s+to\s+help)\b", flags=re.IGNORECASE)
base["ag_palavras_por_turno"] = base["n_palavras_agente"] / base["turnos_agente"].clip(lower=1)

# --- 4. Montagem final e verificação -----------------------------------------
df_features = base
print(f"\ndf_features: {df_features.shape[0]} conversas × {df_features.shape[1]} colunas")
print("Colunas:", list(df_features.columns))
num_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNulos por coluna numérica: {int(df_features[num_cols].isna().sum().sum())} no total")

# --- 5. Prévia de sinal: média das features por classe (churn vs não-churn) ---
print("\n=== Prévia de sinal: média por classe (churn=1 vs churn=0) ===")
prev = df_features.groupby("churn_label")[num_cols].mean().T
prev.columns = ["não-churn", "churn"]
prev["dif_abs"] = (prev["churn"] - prev["não-churn"]).abs()
print(prev.round(3).sort_values("dif_abs", ascending=False).to_string())

print("\nSTEP 6 concluído: df_features pronto — falta juntar as features acústicas (STEP 7, subset de WAVs).")

In [ ]:
# ==============================================================================
# STEP 7: FEATURES ACÚSTICAS — SUBSET DE 160 WAVs (79 churn + 81 não-churn)
# ==============================================================================
# Objetivo: descarregar os WAVs do subset aprovado e extrair features
# acústicas com librosa — globais (conversa inteira) e do CLIENTE (fatia dos
# seus turnos via timestamps, limitada a 120 s para viabilizar o pitch).
#
# Input:  df_features (STEP 6), df_segmentos (STEP 3), ROOT_URL (STEP 1)
# Output: df_acusticas (160 × ~21) e df_modelo (df_features + acústicas)
#
# Features:
#   Globais (ficheiro inteiro):
#     - ac_dur_real_s (validação vs duração declarada), ac_rms_media/std,
#       ac_centroide_media (brilho espectral)
#   Do cliente (só os seus turnos, ≤120 s):
#     - ac_cli_dur_s, ac_cli_rms_media/std (energia), ac_cli_mfcc1..13 (timbre),
#       ac_cli_zcr_media, ac_cli_pitch_media/std (YIN, f0 65–400 Hz),
#       ac_cli_voz_pct (quadros acima do 30.º percentil de RMS)
#
# Nota: ~2,2 GB de download (160 × ~13,8 MB); execução esperada 15–30 min.
# A célula é RETOMÁVEL: conversas já processadas (em runs anteriores
# incompletos) são saltadas automaticamente.
# ==============================================================================

import io
import time
import requests
import numpy as np
import pandas as pd
import librosa

# --- 0. Retoma: preservar resultados de execuções anteriores incompletas -------
resultados = [r for r in globals().get("resultados", []) if isinstance(r, dict)]
ja_feitos = {r.get("id_conversa") for r in resultados}
if "df_acusticas" in globals() and "id_conversa" in getattr(globals().get("df_acusticas"), "columns", []):
    ja_feitos |= set(df_acusticas["id_conversa"])
if ja_feitos:
    print(f"Retoma: {len(ja_feitos)} conversas já processadas serão saltadas.")

# --- 1. Subset: 79 churn + 81 não-churn estratificado por domínio ------------
churn_df = df_features[df_features["churn_label"] == 1]
nao_df = df_features[df_features["churn_label"] == 0]

frac = nao_df["dominio"].value_counts(normalize=True)
n_aloc = (frac * 81).round().astype(int)
amostras = []
for dom, n in n_aloc.items():
    pool = nao_df[nao_df["dominio"] == dom]
    amostras.append(pool.sample(max(1, min(int(n), len(pool))), random_state=SEED))
nao_sel = pd.concat(amostras)
if len(nao_sel) < 81:  # completar se o arredondamento deixou atrás
    extra = nao_df.drop(nao_sel.index).sample(81 - len(nao_sel), random_state=SEED)
    nao_sel = pd.concat([nao_sel, extra])

subset = pd.concat([churn_df, nao_sel]).sample(frac=1, random_state=SEED)  # ordem aleatória
subset = subset.merge(df_conversas[["id_conversa", "ficheiro_wav"]], on="id_conversa", how="left")
assert subset["ficheiro_wav"].isna().sum() == 0, "ficheiro_wav em falta após o merge"
print(f"Subset: {len(subset)} conversas — {int(subset['churn_label'].sum())} churn / "
      f"{int((subset['churn_label'] == 0).sum())} não-churn")
print("Domínios no subset:", dict(subset["dominio"].value_counts()))

# --- 2. Download + extração de features ---------------------------------------
seg_cli = df_segmentos[df_segmentos["role"] == "customer"]
# Caminho real dos WAVs (verificado via API tree + HEAD 200):
# diarization/<pasta_acento>/<file_name>  — o file_name é relativo à pasta do acento
mapa_pasta = df_meta.set_index("file_name")["_pasta_acento"].to_dict()
falhas = []
t0 = time.time()
for k, (idc, ficheiro) in enumerate(subset[["id_conversa", "ficheiro_wav"]].itertuples(index=False), 1):
    if idc in ja_feitos:
        continue
    try:
        resp = requests.get(f"{ROOT_URL}/diarization/{mapa_pasta[ficheiro]}/{ficheiro}", timeout=300)
        resp.raise_for_status()
        y, sr = librosa.load(io.BytesIO(resp.content), sr=None, mono=True)

        reg = {"id_conversa": idc}
        # Globais (conversa inteira)
        reg["ac_dur_real_s"] = len(y) / sr
        rms = librosa.feature.rms(y=y)[0]
        reg["ac_rms_media"] = float(rms.mean())
        reg["ac_rms_std"] = float(rms.std())
        reg["ac_centroide_media"] = float(librosa.feature.spectral_centroid(y=y, sr=sr)[0].mean())

        # Cliente: concatenar os seus turnos via timestamps, limitar a 120 s
        d_cli = seg_cli[seg_cli["id_conversa"] == idc]
        pedacos = [y[int(a * sr):int(b * sr)] for a, b in zip(d_cli["inicio_s"], d_cli["fim_s"])]
        pedacos = [p for p in pedacos if len(p) > 0]
        y_cli = np.concatenate(pedacos) if pedacos else y
        if len(y_cli) / sr > 120:
            y_cli = y_cli[: int(120 * sr)]
        reg["ac_cli_dur_s"] = len(y_cli) / sr

        rms_c = librosa.feature.rms(y=y_cli)[0]
        reg["ac_cli_rms_media"] = float(rms_c.mean())
        reg["ac_cli_rms_std"] = float(rms_c.std())

        mfcc = librosa.feature.mfcc(y=y_cli, sr=sr, n_mfcc=13)
        for i in range(13):
            reg[f"ac_cli_mfcc{i + 1}"] = float(mfcc[i].mean())
        reg["ac_cli_zcr_media"] = float(librosa.feature.zero_crossing_rate(y_cli)[0].mean())

        # Pitch (F0) via YIN nos quadros com voz
        f0 = librosa.yin(y_cli, fmin=65, fmax=400, sr=sr)
        n = min(len(f0), len(rms_c))
        f0, r = f0[:n], rms_c[:n]
        voz = r > np.percentile(r, 30)
        reg["ac_cli_pitch_media"] = float(f0[voz].mean()) if voz.sum() > 10 else np.nan
        reg["ac_cli_pitch_std"] = float(f0[voz].std()) if voz.sum() > 10 else np.nan
        reg["ac_cli_voz_pct"] = float(voz.mean())
        resultados.append(reg)
    except Exception as e:
        falhas.append((idc, str(e)[:100]))
    if k % 20 == 0 or k == len(subset):
        print(f"  {k}/{len(subset)} — {time.time() - t0:.0f} s decorridos, {len(falhas)} falhas")

df_acusticas = pd.DataFrame(resultados)
print(f"\ndf_acusticas: {df_acusticas.shape[0]} conversas × {df_acusticas.shape[1]} colunas")
if falhas:
    print("Falhas (primeiras 5):", falhas[:5])

# --- 3. Verificações -----------------------------------------------------------
chk = df_acusticas.merge(df_conversas[["id_conversa", "duracao_s"]], on="id_conversa")
dif = (chk["ac_dur_real_s"] - chk["duracao_s"]).abs()
print(f"\nDuração real vs declarada: diferença média {dif.mean():.2f} s (max {dif.max():.2f} s)")
print(f"Nulos em df_acusticas: {int(df_acusticas.isna().sum().sum())} (pitch: conversas sem voz detetada)")

# --- 4. Tabela de modelação e prévia de sinal ----------------------------------
df_modelo = df_features.merge(df_acusticas, on="id_conversa", how="inner")
print(f"\ndf_modelo: {df_modelo.shape[0]} conversas × {df_modelo.shape[1]} colunas "
      f"({int(df_modelo['churn_label'].sum())} churn / {int((df_modelo['churn_label'] == 0).sum())} não-churn)")

ac_cols = [c for c in df_acusticas.columns if c.startswith("ac_") and c != "ac_dur_real_s"]
print("\n=== Prévia de sinal acústico (média por classe, top 12 por diferença) ===")
prev = df_modelo.groupby("churn_label")[ac_cols].mean().T
prev.columns = ["não-churn", "churn"]
prev["dif_abs"] = (prev["churn"] - prev["não-churn"]).abs()
print(prev.round(3).sort_values("dif_abs", ascending=False).head(12).to_string())

print("\nSTEP 7 concluído: features acústicas extraídas — df_modelo pronto para o split (STEP 8).")

In [ ]:
# ==============================================================================
# STEP 8: SPLIT ESTRATIFICADO + MODELOS BASELINE (meta: F1 churn >= 0,70)
# ==============================================================================
# Objetivo: treinar modelos baseline para prever o RISCO de churn a partir
# de features comportamentais, acústicas e do agente (NUNCA o texto do
# cliente, que define o label), com 5-fold CV estratificada no treino e
# avaliação final num hold-out de teste 80/20 estratificado.
#
# Input:  df_modelo (STEP 7) — 160 conversas × 54 colunas (79 churn / 81 não)
# Output: métricas F1/AUC (dummy, CV e teste), matriz de confusão, top features
#
# Decisões:
#   - Baseline dummy 'stratified' (adivinha pela prior das classes) p/ contexto
#   - Modelos: Regressão Logística (balanced), Random Forest (400 árvores),
#     Gradient Boosting
#   - Pipeline: StandardScaler (numéricas) + OneHot (categóricas)
#   - Excluídas: id_conversa, churn_label e ac_dur_real_s (duplica duracao_s)
#
# Avisos honestos:
#   - n=160 → teste com 32 conversas (~16 churn): o F1 no teste tem
#     variância considerável; o F1 CV ± desvio é o mais fiável
#   - 'dominio' é metadado legítimo (existe em produção) mas pode capturar o
#     risco do DOMÍNIO (telecom) e não o comportamento por chamada — veremos
#     na importância de features
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- 1. Matriz de features e alvo ---------------------------------------------
EXCLUIR = ["id_conversa", "churn_label", "ac_dur_real_s"]  # id, alvo e duplicada de duracao_s
cat_cols = ["acento", "dominio", "genero_cliente", "genero_agente"]
X = df_modelo.drop(columns=[c for c in EXCLUIR if c in df_modelo.columns])
y = df_modelo["churn_label"]
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"Features: {len(num_cols)} numéricas + {len(cat_cols)} categóricas | n={len(X)} | "
      f"churn={int(y.sum())} ({y.mean():.0%})")

# --- 2. Split estratificado 80/20 ----------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print(f"Treino: {len(X_train)} ({int(y_train.sum())} churn) | "
      f"Teste: {len(X_test)} ({int(y_test.sum())} churn)")

# --- 3. Baseline dummy (contexto para o F1) ------------------------------------
dummy = DummyClassifier(strategy="stratified", random_state=SEED)
dummy.fit(X_train, y_train)
f1_dummy = f1_score(y_test, dummy.predict(X_test))
print(f"\nBaseline dummy (estratificado, i.e., adivinhar): F1 no teste = {f1_dummy:.3f}")

# --- 4. Modelos e 5-fold CV estratificada no treino -----------------------------
preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
modelos = {
    "Regressão Logística": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

resultados_cv = {}
for nome, clf in modelos.items():
    pipe = Pipeline([("prep", preproc), ("modelo", clf)])
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=("f1", "roc_auc"))
    resultados_cv[nome] = res["test_f1"].mean()
    print(f"{nome:22s} F1 CV = {res['test_f1'].mean():.3f} ± {res['test_f1'].std():.3f} | "
          f"AUC CV = {res['test_roc_auc'].mean():.3f}")

# --- 5. Melhor modelo (por F1 CV) avaliado no teste ----------------------------
melhor_nome = max(resultados_cv, key=resultados_cv.get)
melhor_clf = modelos[melhor_nome]
pipe_final = Pipeline([("prep", preproc), ("modelo", melhor_clf)])
pipe_final.fit(X_train, y_train)
y_pred = pipe_final.predict(X_test)
y_proba = pipe_final.predict_proba(X_test)[:, 1]

f1_teste = f1_score(y_test, y_pred)
auc_teste = roc_auc_score(y_test, y_proba)
print(f"\n=== Melhor modelo: {melhor_nome} — avaliação no TESTE ===")
print(classification_report(y_test, y_pred, target_names=["não-churn", "churn"], digits=3))
print("Matriz de confusão (linhas=real, colunas=previsto):")
print(confusion_matrix(y_test, y_pred))
print(f"\nF1 (churn) no teste: {f1_teste:.3f} | AUC no teste: {auc_teste:.3f}")
print(f"Meta F1 >= {F1_MINIMO_ACEITAVEL}: {'ATINGIDA' if f1_teste >= F1_MINIMO_ACEITAVEL else 'NÃO atingida'} "
      f"| dummy: {f1_dummy:.3f}")

# --- 6. Top features do melhor modelo -------------------------------------------
nomes_feat = pipe_final.named_steps["prep"].get_feature_names_out()
if hasattr(melhor_clf, "feature_importances_"):
    imp = melhor_clf.feature_importances_
else:
    imp = np.abs(melhor_clf.coef_[0])
top = pd.Series(imp, index=nomes_feat).sort_values(ascending=False).head(15)
print("\n=== Top 15 features ===")
print(top.round(4).to_string())

print("\nSTEP 8 concluído: baseline treinado e avaliado — pronto para a análise final (STEP 9).")

In [ ]:
# ==============================================================================
# STEP 8b: MELHORIA DO MODELO — SPLIT POR FALANTE + TUNING + MENOS METADADOS
# ==============================================================================
# Objetivo: eliminar a possível memorização de vozes (CV e hold-out
# AGRUPADOS pelo falante do CLIENTE), reduzir a dependência de metadados
# (dominio/acento) e afinar hiperparâmetros — comparando sempre com o
# baseline do STEP 8.
#
# Input:  df_modelo (STEP 7), df_segmentos (STEP 3)
# Output: F1 CV por variante (agrupada), tuning, hold-out final DISJOINT POR
#         FALANTE, top features, comparação com o STEP 8
#
# Variantes de features (Regressão Logística, class_weight balanced):
#   V1 — todas (igual ao STEP 8, mas CV agrupada por falante)
#   V2 — sem acento (acento ≈ proxy do falante)
#   V3 — sem acento nem domínio (só comportamento + acústica + agente)
# Tuning: LR (grid de C) e RF (profundidade/folhas) na melhor variante
#
# Nota honesta: se mesmo assim o F1 não chegar a 0,70, a conclusão é que o
# sinal áudio/comportamento é limitado face a um label textual (proxy).
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

F1_STEP8_CV = 0.609      # baseline STEP 8: F1 CV por conversa (LR, todas)
F1_STEP8_TESTE = 0.552   # baseline STEP 8: F1 no teste por conversa

# --- 1. Grupo = falante do CLIENTE (a voz que os MFCCs memorizariam) -----------
cli_speaker = (df_segmentos[df_segmentos["role"] == "customer"]
               .groupby("id_conversa")["speaker_id"].first())
df_modelo_g = df_modelo.merge(cli_speaker.rename("speaker_cliente"), on="id_conversa", how="left")
assert df_modelo_g["speaker_cliente"].isna().sum() == 0, "speaker_cliente em falta"

grupos = df_modelo_g["speaker_cliente"].values
por_speaker = pd.Series(grupos).value_counts()
print(f"Falantes do cliente no subset: {por_speaker.size} únicos para {len(df_modelo_g)} conversas")
print(f"Máx. conversas por falante: {por_speaker.max()} | mediana: {por_speaker.median():.0f}")

# --- 2. Variantes de features — LR com CV AGRUPADA por falante -----------------
EXCLUIR = ["id_conversa", "churn_label", "ac_dur_real_s", "speaker_cliente"]
CAT_TODAS = ["acento", "dominio", "genero_cliente", "genero_agente"]
y = df_modelo_g["churn_label"]

def pipeline_de(cat_usar, clf):
    drop = set(EXCLUIR) | (set(CAT_TODAS) - set(cat_usar))
    X = df_modelo_g.drop(columns=[c for c in df_modelo_g.columns if c in drop])
    num = [c for c in X.columns if c not in cat_usar]
    pre = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_usar),
    ])
    return X, Pipeline([("prep", pre), ("modelo", clf)])

VARIANTES = {
    "V1 todas":               CAT_TODAS,
    "V2 sem acento":          ["dominio", "genero_cliente", "genero_agente"],
    "V3 sem acento/domínio":  ["genero_cliente", "genero_agente"],
}
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"\n=== LR com CV AGRUPADA por falante (STEP 8, por conversa: {F1_STEP8_CV:.3f}) ===")
melhor = (None, -1.0)
for nome, cats in VARIANTES.items():
    Xv, pipe = pipeline_de(cats, LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    res = cross_validate(pipe, Xv, y, cv=sgkf, groups=grupos, scoring=("f1", "roc_auc"))
    f1m, f1s, auc = res["test_f1"].mean(), res["test_f1"].std(), res["test_roc_auc"].mean()
    print(f"  {nome:22s} F1 CV = {f1m:.3f} ± {f1s:.3f} | AUC CV = {auc:.3f}")
    if f1m > melhor[1]:
        melhor = (nome, f1m)
melhor_var = melhor[0]
print(f"Melhor variante: {melhor_var}")

# --- 3. Tuning (LR e RF) na melhor variante, CV agrupada ------------------------
X_best, pipe_lr = pipeline_de(VARIANTES[melhor_var],
                               LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED))
grelha_lr = GridSearchCV(pipe_lr, {"modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
                         cv=sgkf, scoring="f1")
grelha_lr.fit(X_best, y, groups=grupos)
print(f"\nLR tuning: melhor C = {grelha_lr.best_params_['modelo__C']} | F1 CV = {grelha_lr.best_score_:.3f}")

_, pipe_rf = pipeline_de(VARIANTES[melhor_var],
                          RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED))
grelha_rf = GridSearchCV(pipe_rf, {"modelo__max_depth": [3, 5, None], "modelo__min_samples_leaf": [1, 3, 5]},
                         cv=sgkf, scoring="f1")
grelha_rf.fit(X_best, y, groups=grupos)
print(f"RF tuning: {grelha_rf.best_params_} | F1 CV = {grelha_rf.best_score_:.3f}")

# --- 4. Avaliação final: hold-out DISJOINT POR FALANTE --------------------------
melhor_grelha = grelha_lr if grelha_lr.best_score_ >= grelha_rf.best_score_ else grelha_rf
melhor_nome = "Regressão Logística (tuning)" if melhor_grelha is grelha_lr else "Random Forest (tuning)"

idx_treino, idx_teste = next(sgkf.split(X_best, y, groups=grupos))
X_tr, X_te = X_best.iloc[idx_treino], X_best.iloc[idx_teste]
y_tr, y_te = y.iloc[idx_treino], y.iloc[idx_teste]
spk_tr = set(pd.Series(grupos).iloc[idx_treino])
spk_te = set(pd.Series(grupos).iloc[idx_teste])
print(f"\nHold-out por falante — treino {len(X_tr)} ({int(y_tr.sum())} churn, {len(spk_tr)} falantes) / "
      f"teste {len(X_te)} ({int(y_te.sum())} churn, {len(spk_te)} falantes)")
print(f"Falantes partilhados treino∩teste: {len(spk_tr & spk_te)} (tem de ser 0)")

final = melhor_grelha.best_estimator_
final.fit(X_tr, y_tr)
y_pred = final.predict(X_te)
y_proba = final.predict_proba(X_te)[:, 1]
f1_final = f1_score(y_te, y_pred)
auc_final = roc_auc_score(y_te, y_proba)

print(f"\n=== FINAL: {melhor_nome} — hold-out DISJOINT POR FALANTE ===")
print(classification_report(y_te, y_pred, target_names=["não-churn", "churn"], digits=3))
print("Matriz de confusão (linhas=real, colunas=previsto):")
print(confusion_matrix(y_te, y_pred))
print(f"\nF1 (churn): {f1_final:.3f} | AUC: {auc_final:.3f}")
print(f"Comparação: STEP 8 teste {F1_STEP8_TESTE:.3f} → STEP 8b {f1_final:.3f} | "
      f"meta {F1_MINIMO_ACEITAVEL}: {'ATINGIDA' if f1_final >= F1_MINIMO_ACEITAVEL else 'NÃO atingida'}")

# --- 5. Top features do modelo final ---------------------------------------------
clf_final = final.named_steps["modelo"]
nomes_feat = final.named_steps["prep"].get_feature_names_out()
imp = clf_final.feature_importances_ if hasattr(clf_final, "feature_importances_") else np.abs(clf_final.coef_[0])
top = pd.Series(imp, index=nomes_feat).sort_values(ascending=False).head(15)
print("\n=== Top 15 features (modelo final) ===")
print(top.round(4).to_string())

print("\nSTEP 8b concluído: melhoria testada com separação rigorosa por falante.")

In [ ]:
# ==============================================================================
# STEP 9: ANÁLISE FINAL — SÍNTESE, PERGUNTAS DE NEGÓCIO E CONCLUSÕES
# ==============================================================================
# Objetivo: fechar o projeto com respostas às perguntas de negócio e uma
# avaliação do modelo como RANKER de priorização (uso recomendado), o
# perfil comportamental do cliente em risco, e conclusões/limitações honestas.
#
# Input:  df_conversas (873), df_modelo (160), modelo final + hold-out (STEP 8b)
# Output: taxa de churn por domínio; ganhos de priorização (top-k); perfil do
#         cliente em risco; síntese final com números verificados
# ==============================================================================

import numpy as np
import pandas as pd

# --- 1. Pergunta 1: quais os domínios com maior risco de churn? (873 convs) ----
taxa = (df_conversas.groupby("dominio")["churn_label"].agg(["sum", "count", "mean"])
        .sort_values("mean", ascending=False))
taxa.columns = ["churn", "conversas", "taxa"]
print("=== Risco de churn por domínio (873 conversas) ===")
print(taxa.round(3).to_string())
print(f"\nTaxa global: {df_conversas['churn_label'].mean():.1%} "
      f"({int(df_conversas['churn_label'].sum())}/873 conversas)")

# --- 2. Pergunta 2: o modelo funciona como RANKER de priorização? -------------
# Hold-out DISJOINT POR FALANTE (42 conversas, 25 churn): se a equipa só
# conseguir tratar as k chamadas mais arriscadas, quantos casos apanha?
df_rank = (pd.DataFrame({"prob": y_proba, "real": y_te.values})
           .sort_values("prob", ascending=False).reset_index(drop=True))
n_churn = int(df_rank["real"].sum())
base = df_rank["real"].mean()
print(f"\n=== Priorização no hold-out por falante (42 convs, {n_churn} churn, taxa base {base:.1%}) ===")
print(f"{'top-k':>6} | {'churn apanhados':>16} | {'precisão':>9} | {'recall':>7} | {'lift':>5}")
for k in (5, 10, 15, 21):
    hits = int(df_rank["real"].iloc[:k].sum())
    prec = hits / k
    rec = hits / n_churn
    print(f"{k:>6} | {hits:>7}/{k:<8} | {prec:>9.1%} | {rec:>7.1%} | {prec / base:>5.1f}x")

# --- 3. Pergunta 3: qual o perfil do cliente em risco? (160 convs) -------------
cols_perfil = ["n_palavras_cliente", "dur_fala_cliente_s", "interrupcoes_cliente",
               "latencia_media_cliente_s", "turnos_cliente", "pct_silencio",
               "ag_pedidos_desculpa", "ag_empaticas", "ac_cli_pitch_media",
               "ac_centroide_media", "ritmo_palavras_cliente", "ac_cli_dur_s"]
perfil = df_modelo.groupby("churn_label")[cols_perfil].mean().T
perfil.columns = ["não-churn", "churn"]
perfil["dif_rel"] = (perfil["churn"] - perfil["não-churn"]) / perfil["não-churn"]
print("\n=== Perfil do cliente em risco (médias no subset de 160) ===")
print(perfil.round(2).sort_values("dif_rel", key=abs, ascending=False).to_string())

# --- 4. Síntese final (números verificados nas STEP 1–8b) ---------------------
print("""
=============================== SÍNTESE FINAL ===============================
DADOS      : AppTek Callcenter Dialogues — 873 conversas role-played (151,1 h),
            16 domínios, 14 acentos, 94 679 segmentos com diarização.

LABEL      : 'risco de churn' (proxy linguístico: cancelamento/encerramento,
            frustração forte, escalada, ameaça de saída) — 79/873 (9,0%).
            O TEXTO do cliente NUNCA é feature (só define o label).

MODELO     : Regressão Logística (C=0,03), 51 features comportamentais,
            acústicas e do agente — avaliado em hold-out DISJOINT POR
            FALANTE (42 convs, 16 falantes nunca vistos):
              F1 churn 0,698 | AUC 0,708 | precisão 0,833 | recall 0,600

EVOLUÇÃO   : baseline (STEP 8, split por conversa)  F1 0,552
            → split por falante + tuning (STEP 8b)   F1 0,698
            META F1 >= 0,70: não atingida por 0,002 (no limiar, n pequeno).

LICÕES     : 1) split por conversa inflava o F1 em ~0,06 (memorização de
               vozes — MFCCs + acento);
            2) sem regularização, o modelo decorava domínio/acento; com
               C=0,03 lê comportamento: MFCC5, duração da fala do cliente,
               latência, ritmo, interrupções, pedidos de desculpa do agente;
            3) o sinal áudio/comportamento existe mas é moderado — o label
               deriva de TEXTO que não vemos (por opção anti-fuga).

USO        : RANKER de priorização (AUC ~0,71): sinalizar as chamadas de
            maior score para intervenção/follow-up — quando sinaliza,
            acerta ~83%.

LIMITAÇÕES : dataset role-played (não clientes reais); label proxy;
            160/873 conversas com áudio processado; n pequeno → intervalos
            largos; generalização a domínios/acentos novos por confirmar.

PRÓXIMOS   : processar os restantes 713 WAVs; treino multi-modal (áudio
PASSOS      + texto em setup com label externo); validar em dados reais.
=============================================================================""")

print("\nSTEP 9 concluído: análise final apresentada — projeto fechado.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 1/7 — Em que domínios se concentra o risco de churn?
# ==============================================================================
# Decisão a que responde: onde priorizar formação de agentes e monitorização
# de qualidade das equipas.
# Dados: 873 conversas (df_conversas; label proxy do STEP 5).
import pandas as pd

taxa_dom = (df_conversas.groupby("dominio")["churn_label"]
            .agg(["sum", "count", "mean"]).sort_values("mean", ascending=False))
taxa_dom.columns = ["churn", "conversas", "taxa"]
print(taxa_dom.round(3).to_string())
glob = df_conversas["churn_label"].mean()
print(f"\nTaxa global: {glob:.1%} ({int(df_conversas['churn_label'].sum())}/873 conversas)")

top3 = taxa_dom.head(3).reset_index()
baixo2 = taxa_dom.tail(2).reset_index()
print(f"\nRESPOSTA: o risco concentra-se em "
      + "; ".join(f"{r.dominio} ({r.taxa:.1%})" for r in top3.itertuples())
      + f", acima da taxa global de {glob:.1%}; os domínios mais calmos são "
      + "; ".join(f"{r.dominio} ({r.taxa:.1%})" for r in baixo2.itertuples()) + ".")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 2/7 — Qual o perfil ACÚSTICO do cliente em risco?
# ==============================================================================
# Decisão a que responde: que sinais vocais (energia, pitch, timbre) usar numa
# monitorização de qualidade de chamada em tempo quase real.
# Dados: 160 conversas com áudio processado (df_modelo; features ac_*).
import pandas as pd

cols_ac = [c for c in df_modelo.columns if c.startswith("ac_") and c != "ac_dur_real_s"]
perf_ac = df_modelo.groupby("churn_label")[cols_ac].mean().T
perf_ac.columns = ["não-churn", "churn"]
perf_ac["dif_rel%"] = (perf_ac["churn"] - perf_ac["não-churn"]) / perf_ac["não-churn"] * 100
perf_ac = perf_ac.sort_values("dif_rel%", key=abs, ascending=False)
print(perf_ac.round(2).to_string())

top3 = perf_ac.head(3)
print("\nRESPOSTA: as maiores diferenças acústicas são "
      + ", ".join(f"{i} ({v:+.0f}%)" for i, v in top3["dif_rel%"].items())
      + f". O centróide espectral sobe de {perf_ac.loc['ac_centroide_media', 'não-churn']:.0f} para "
      f"{perf_ac.loc['ac_centroide_media', 'churn']:.0f} Hz e o pitch do cliente de "
      f"{perf_ac.loc['ac_cli_pitch_media', 'não-churn']:.0f} para "
      f"{perf_ac.loc['ac_cli_pitch_media', 'churn']:.0f} Hz — voz mais brilhante e aguda no cliente em risco.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 3/7 — Como se COMPORTA o cliente em risco (duração, ritmo,
# interrupções)?
# ==============================================================================
# Decisão a que responde: que indicadores comportamentais as equipas devem
# reconhecer durante a chamada.
# Dados: 160 conversas (df_modelo).
import pandas as pd

cols_comp = ["duracao_s", "tempo_fala_s", "dur_fala_cliente_s", "n_palavras_cliente",
             "ritmo_palavras_cliente", "interrupcoes_cliente", "turnos_cliente",
             "densidade_turnos", "latencia_media_cliente_s", "pct_silencio"]
perf_c = df_modelo.groupby("churn_label")[cols_comp].mean().T
perf_c.columns = ["não-churn", "churn"]
perf_c["dif_rel%"] = (perf_c["churn"] - perf_c["não-churn"]) / perf_c["não-churn"] * 100
perf_c = perf_c.sort_values("dif_rel%", key=abs, ascending=False)
print(perf_c.round(2).to_string())

print(f"\nRESPOSTA: o cliente em risco interrompe mais "
      f"({perf_c.loc['interrupcoes_cliente', 'não-churn']:.1f} → {perf_c.loc['interrupcoes_cliente', 'churn']:.1f}, "
      f"{perf_c.loc['interrupcoes_cliente', 'dif_rel%']:+.0f}%), responde mais depressa "
      f"(latência {perf_c.loc['latencia_media_cliente_s', 'não-churn']:.2f} → "
      f"{perf_c.loc['latencia_media_cliente_s', 'churn']:.2f} s) e deixa menos silêncio "
      f"({perf_c.loc['pct_silencio', 'não-churn']:.2f} → {perf_c.loc['pct_silencio', 'churn']:.2f}) "
      "— é um cliente ativo e rápido, não calado.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 4/7 — O comportamento do AGENTE difere em chamadas de risco?
# ==============================================================================
# Decisão a que responde: 'pedidos de desculpa/empatia' pode servir de métrica
# de qualidade da equipa?
# Dados: 160 conversas (df_modelo; features ag_* e do lado do agente).
import pandas as pd

cols_ag = ["ag_pedidos_desculpa", "ag_agradecimentos", "ag_empaticas",
           "ag_palavras_por_turno", "interrupcoes_agente", "turnos_agente",
           "dur_media_turno_agente_s", "n_palavras_agente", "ratio_palavras_agente_cliente"]
perf_a = df_modelo.groupby("churn_label")[cols_ag].mean().T
perf_a.columns = ["não-churn", "churn"]
perf_a["dif_rel%"] = (perf_a["churn"] - perf_a["não-churn"]) / perf_a["não-churn"] * 100
perf_a = perf_a.sort_values("dif_rel%", key=abs, ascending=False)
print(perf_a.round(2).to_string())

print(f"\nRESPOSTA: o agente pede desculpa {perf_a.loc['ag_pedidos_desculpa', 'não-churn']:.1f}× por chamada "
      f"normal vs {perf_a.loc['ag_pedidos_desculpa', 'churn']:.1f}× em chamadas de risco "
      f"({perf_a.loc['ag_pedidos_desculpa', 'dif_rel%']:+.0f}%) — desculpas e empatia ACOMPANHAM a fricção "
      "(são reação ao problema, não prevenção): não servem como métrica isolada de qualidade.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 5/7 — É possível prever o risco SEM analisar o texto do cliente?
# ==============================================================================
# Decisão a que responde: pode um modelo de áudio/comportamento triar chamadas
# sem depender da transcrição do cliente?
# Dados: modelo final (LR C=0,03) no hold-out DISJOINT POR FALANTE (42 conversas).
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

y_pred5 = final.predict(X_te)
y_proba5 = final.predict_proba(X_te)[:, 1]
f1_5 = f1_score(y_te, y_pred5)
pr_5 = precision_score(y_te, y_pred5)
rc_5 = recall_score(y_te, y_pred5)
auc_5 = roc_auc_score(y_te, y_proba5)
print(f"F1 churn: {f1_5:.3f} (meta {F1_MINIMO_ACEITAVEL}) | AUC: {auc_5:.3f} | "
      f"precisão: {pr_5:.1%} | recall: {rc_5:.1%}")
print("Matriz de confusão (linhas=real, colunas=previsto):")
print(confusion_matrix(y_te, y_pred5))

print(f"\nRESPOSTA: sim, com sinal moderado — F1 {f1_5:.3f} (meta {F1_MINIMO_ACEITAVEL}, falha por "
      f"{F1_MINIMO_ACEITAVEL - f1_5:.3f}) e AUC {auc_5:.3f} com falantes nunca vistos; quando sinaliza risco, "
      f"acerta {pr_5:.0%}. O texto do cliente NÃO é feature — só comportamento, áudio e texto do agente.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 6/7 — Vale a pena usar o modelo para PRIORIZAR intervenções?
# ==============================================================================
# Decisão a que responde: quantas chamadas tratar primeiro e quantos casos reais
# de risco se apanham com essa fila.
# Dados: ranking do hold-out por falante (42 conversas, 25 churn).
import pandas as pd

rk = (pd.DataFrame({"prob": y_proba, "real": y_te.values})
      .sort_values("prob", ascending=False).reset_index(drop=True))
n_churn = int(rk["real"].sum())
base = rk["real"].mean()
print(f"Taxa base de churn no hold-out: {base:.1%} ({n_churn}/{len(rk)})\n")
print(f"{'top-k':>6} | {'churn apanhados':>16} | {'precisão':>9} | {'recall':>7} | {'lift':>6}")
for k in (5, 10, 15, 21):
    hits = int(rk["real"].iloc[:k].sum())
    prec = hits / k
    print(f"{k:>6} | {hits:>7}/{k:<8} | {prec:>9.1%} | {hits / n_churn:>7.1%} | {prec / base:>5.1f}x")

hits21 = int(rk["real"].iloc[:21].sum())
hits5 = int(rk["real"].iloc[:5].sum())
print(f"\nRESPOSTA: sim, como filtro LARGO: o top-50% apanha {hits21}/{n_churn} casos "
      f"({hits21 / 21:.0%} de precisão, lift {hits21 / 21 / base:.1f}x); mas o top-5 extremo é fraco "
      f"({hits5}/5) — usar para ordenar uma fila de follow-up, não para escolher só 5 chamadas.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 7/7 — Qual o LIMIAR operacional para sinalizar follow-up?
# ==============================================================================
# Decisão a que responde: a partir de que score encaminhar uma chamada para a
# equipa de retenção, equilibrando tamanho da fila e casos perdidos.
# Dados: probabilidades do modelo final no hold-out por falante (42 conversas).
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

limiares = [0.30, 0.40, 0.50, 0.60, 0.70]
linhas = []
for t in limiares:
    sel = (y_proba >= t).astype(int)
    if sel.sum() == 0:
        continue
    linhas.append({
        "limiar": t,
        "sinalizadas": int(sel.sum()),
        "precisão": precision_score(y_te, sel, zero_division=0),
        "recall": recall_score(y_te, sel),
        "F1": f1_score(y_te, sel),
    })
tab = pd.DataFrame(linhas)
print(tab.round(3).to_string(index=False))

melhor = tab.loc[tab["F1"].idxmax()]
r50 = tab.loc[(tab["limiar"] - 0.50).abs().idxmin()]
print(f"\nRESPOSTA: o limiar {melhor['limiar']:.2f} maximiza o F1 ({melhor['F1']:.3f}); "
      f"operacionalmente, p >= 0,50 sinaliza {int(r50['sinalizadas'])} de {len(y_te)} chamadas com "
      f"{r50['precisão']:.0%} de precisão e {r50['recall']:.0%} de recall — um bom compromisso "
      "entre tamanho da fila e casos perdidos.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 8/14 — Qual o MOTIVO de cancelamento mais frequente nas
# transcrições?
# ==============================================================================
# Decisão a que responde: que causas atacar primeiro (preço, produto,
# logística, atendimento) para reduzir cancelamentos.
# Dados: texto do cliente nas 79 conversas de risco (de 873).
# Nota honesta: o dataset NÃO tem campo 'motivo' — categorização PROXY por
# palavras-chave, com sobreposição possível entre motivos.
# Leakage: análise descritiva do texto do cliente — NÃO entra no modelo.
import pandas as pd
import re

cli = df_segmentos[df_segmentos["role"] == "customer"][["id_conversa", "texto"]]
churn_ids = set(df_conversas.loc[df_conversas["churn_label"] == 1, "id_conversa"])
texto_churn = (cli[cli["id_conversa"].isin(churn_ids)]
               .groupby("id_conversa")["texto"].apply(" ".join).str.lower())
n = len(texto_churn)
print(f"Conversas de risco analisadas: {n}\n")

MOTIVOS = {
    "preço/custos":        r"\b(price|fee|charge|expensive|cost|billing|payment)\b",
    "problema de serviço": r"\b(broken|faulty|defect|not work|doesn't work|stopped working|error|issue|problem)\b",
    "demora/atraso":       r"\b(wait|delay|late|waiting|slow)\b",
    "mudança de operador": r"\b(switch|another provider|competitor|elsewhere|other company)\b",
    "atendimento ruim":    r"\b(rude|unhelpful|useless|incompetent|supervisor|speak to a manager)\b",
    "reembolso/dinheiro":  r"\b(refund|reimburse|money back|compensation)\b",
}
res = {m: int(texto_churn.str.contains(rx, regex=True).sum()) for m, rx in MOTIVOS.items()}
mot = pd.Series(res).sort_values(ascending=False)
print("Conversas de risco que mencionam cada motivo (com sobreposição):")
print(mot.to_string())

algum = texto_churn.apply(lambda t: any(re.search(rx, t) for rx in MOTIVOS.values()))
print(f"\nSem motivo identificado por palavras-chave: {int((~algum).sum())}/{n}")
top2 = mot.head(2)
print(f"\nRESPOSTA: os motivos mais frequentes são "
      + " e ".join(f"{m} ({v} conversas, {v / n:.0%})" for m, v in top2.items())
      + " — categorização proxy por palavras-chave no texto do cliente, com sobreposição possível.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 9/14 — Que PALAVRAS o cliente usa antes de desistir?
# ==============================================================================
# Decisão a que responde: treinar agentes para reconhecer o vocabulário de
# despedida/finalização antes de o cliente cortar a relação.
# Dados: ÚLTIMOS 3 turnos do cliente por conversa (873) — churn vs não-churn.
# Leakage: análise descritiva — NÃO entra no modelo.
import pandas as pd
import re
from collections import Counter

STOP = set("the a an and or but if then so to of in on at for with about is am are was were be been being i you he she it we they me my mine your yours our their this that these those do does did have has had will would can could should just not no yes okay ok like know get got go going really right now well please thank thanks hello hi yeah one also very as because from by what when where how there here".split())

cli = df_segmentos[df_segmentos["role"] == "customer"].sort_values("inicio_s")
ultimos = (cli.groupby("id_conversa").tail(3)
            .groupby("id_conversa")["texto"].apply(" ".join).str.lower())
rot = df_conversas.set_index("id_conversa")["churn_label"]
ids_churn = [i for i in ultimos.index if rot[i] == 1]
ids_nao = [i for i in ultimos.index if rot[i] == 0]

def contar(ids):
    c = Counter()
    for i in ids:
        c.update(w for w in re.findall(r"[a-z']+", ultimos[i]) if w not in STOP and len(w) > 2)
    return c

c_churn, c_nao = contar(ids_churn), contar(ids_nao)
linhas = []
for w, k in c_churn.items():
    if k < 6:
        continue
    tx_c = k / len(ids_churn)
    tx_n = c_nao.get(w, 0) / len(ids_nao)
    linhas.append({"palavra": w, "churn": k, "taxa churn": tx_c, "taxa não-churn": tx_n,
                   "enriquecimento": tx_c / max(tx_n, 1 / len(ids_nao))})
tab = (pd.DataFrame(linhas).query("enriquecimento >= 3")
       .sort_values("taxa churn", ascending=False).head(15))
if tab.empty:
    print("Nenhuma palavra cumpre o critério de enriquecimento (>=3x) nos turnos finais.")
else:
    show = tab.copy()
    show["taxa churn"] = (show["taxa churn"] * 100).round(0).astype(int).astype(str) + "%"
    show["taxa não-churn"] = (show["taxa não-churn"] * 100).round(1).astype(str) + "%"
    print("Palavras características dos ÚLTIMOS turnos do cliente (churn vs não-churn):")
    print(show.to_string(index=False))
    lista = ", ".join(f"'{w}' ({t:.0%})" for w, t in zip(tab.head(6)["palavra"], tab.head(6)["taxa churn"]))
    print(f"\nRESPOSTA: nos turnos finais das {len(ids_churn)} conversas de risco, destacam-se "
          f"{lista} — muito mais frequentes nas desistências do que nas conversas normais.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 10/14 — Os atendimentos mais LONGOS têm mais churn?
# ==============================================================================
# Decisão a que responde: deve o supervisor cortar/escalar chamadas longas?
# Dados: 873 conversas (duração vs churn_label).
import pandas as pd
from scipy.stats import pointbiserialr

d = df_conversas[["duracao_s", "churn_label"]].dropna()
r, p = pointbiserialr(d["churn_label"], d["duracao_s"])
m_ch = d.loc[d["churn_label"] == 1, "duracao_s"].mean()
m_nc = d.loc[d["churn_label"] == 0, "duracao_s"].mean()
print(f"Correlação duração ↔ churn: r = {r:.3f} (p-valor = {p:.3f}) | n = {len(d)}")
print(f"Duração média: churn {m_ch:.0f} s vs não-churn {m_nc:.0f} s (diferença {(m_ch - m_nc) / m_nc:+.1%})\n")

d = d.copy()
d["quintil"] = pd.qcut(d["duracao_s"], 5, labels=["1º (curtas)", "2º", "3º", "4º", "5º (longas)"])
tab = d.groupby("quintil", observed=True).agg(
    conversas=("churn_label", "count"), churn=("churn_label", "sum"),
    taxa=("churn_label", "mean"), dur_media=("duracao_s", "mean"))
print(tab.round(3).to_string())

if p >= 0.05:
    print(f"\nRESPOSTA: não — a correlação é estatisticamente nula (r = {r:.3f}, p = {p:.3f}): "
          f"chamadas de risco duram em média {m_ch:.0f} s vs {m_nc:.0f} s ({(m_ch - m_nc) / m_nc:+.1%}). "
          "A duração por si só NÃO é sinal de churn — o que diferencia é o comportamento durante a chamada.")
else:
    print(f"\nRESPOSTA: sim, existe correlação significativa (r = {r:.3f}, p = {p:.3f}), mas fraca: "
          f"chamadas de risco duram {m_ch:.0f} s vs {m_nc:.0f} s em média.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 11/14 — Há correlação entre TOM DE VOZ frustrado e
# cancelamento?
# ==============================================================================
# Tom 'frustrado' aproximado por marcadores acústicos: pitch (altura e
# variação), energia (RMS), variabilidade, brilho (centróide) e ZCR.
# Dados: 160 conversas com features acústicas (df_modelo).
import pandas as pd
from scipy.stats import pointbiserialr

cols_tom = ["ac_cli_pitch_media", "ac_cli_pitch_std", "ac_cli_rms_media", "ac_cli_rms_std",
            "ac_cli_zcr_media", "ac_centroide_media", "ac_rms_media", "ac_rms_std"]
linhas = []
for c in cols_tom:
    sub = df_modelo[["churn_label", c]].dropna()
    rr, pp = pointbiserialr(sub["churn_label"], sub[c])
    linhas.append({"marcador": c, "r": rr, "p-valor": pp,
                   "média não-churn": sub.loc[sub["churn_label"] == 0, c].mean(),
                   "média churn": sub.loc[sub["churn_label"] == 1, c].mean()})
tab = pd.DataFrame(linhas).sort_values("r", key=abs, ascending=False)
print("Marcadores acústicos vs churn (160 conversas; correlação ponto-bisserial):")
print(tab.round(3).to_string(index=False))

sig = tab[tab["p-valor"] < 0.05]
if len(sig):
    txt = " e ".join(f"{m} (r = {rr:+.2f})" for m, rr in zip(sig["marcador"], sig["r"]))
    print(f"\nRESPOSTA: sim, mas ligeira — correlações significativas (p < 0,05) em {txt}: voz mais "
          "aguda/brilhante e mais energética acompanha o risco. Contudo cada marcador isolado é fraco "
          "(|r| < 0,3): é a COMBINAÇÃO de sinais, como o modelo usa, que gera valor (AUC 0,71).")
else:
    print(f"\nRESPOSTA: as correlações são todas fracas e nenhuma é significativa a p < 0,05 "
          f"(máx |r| = {tab['r'].abs().max():.2f}) — individualmente nenhum marcador de tom prevê o "
          "cancelamento; apenas a combinação dos sinais (o modelo) tem valor preditivo (AUC 0,71).")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 12/14 — Qual o DEPARTAMENTO com mais reclamações?
# ==============================================================================
# 'Departamento' ≈ domínio. 'Reclamação' ≈ proxy: pedidos de reembolso/
# compensação ou menção explícita a complaint no texto do cliente.
import pandas as pd
import re

cli = df_segmentos[df_segmentos["role"] == "customer"][["id_conversa", "texto"]]
tem_recl = (cli.groupby("id_conversa")["texto"]
            .apply(lambda s: bool(re.search(
                r"\b(refund|reimburse|money back|compensation|chargeback|complain|complaint)\b",
                " ".join(s), re.I))))
df_conversas["reclamacao"] = df_conversas["id_conversa"].map(tem_recl)
tab = df_conversas.groupby("dominio").agg(
    reclamacoes=("reclamacao", "sum"), conversas=("reclamacao", "count"))
tab["taxa"] = tab["reclamacoes"] / tab["conversas"]
tab = tab.sort_values("reclamacoes", ascending=False)
print("Reclamações (pedidos de reembolso/compensação ou complaint) por domínio (873 conversas):")
print(tab.round(3).to_string())

top = tab.iloc[0]
print(f"\nRESPOSTA: o domínio com mais reclamações é {top.name} — {int(top['reclamacoes'])} de "
      f"{int(top['conversas'])} conversas ({top['taxa']:.0%}); proxy: menções a refund/compensação/"
      "complaint no texto do cliente. Nota: reclamar NÃO é o mesmo que estar em risco de churn.")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 13/14 — Que HORÁRIOS têm pior satisfação?
# ==============================================================================
# Regra do projeto: verificar primeiro se os dados existem — nunca inventar.
import pandas as pd

print("Colunas dos metadados (df_meta):", list(df_meta.columns))
print("Colunas dos segmentos (df_segmentos):", list(df_segmentos.columns))
print("\nInício de uma conversa (start/end dos primeiros segmentos):")
print(df_segmentos[["id_conversa", "inicio_s", "fim_s", "role"]].head(4).to_string(index=False))

print("\nRESPOSTA: não é possível responder com este dataset — não existe qualquer registo de "
      "DATA/HORA das chamadas no AppTek Callcenter Dialogues. Os campos start/end dos segmentos são "
      "offsets DENTRO do áudio (posição na conversa, começando perto de 0), não o horário do "
      "atendimento. Recomendação para produção: registar o timestamp das chamadas para permitir "
      "análise de satisfação por hora do dia (ex.: detetar picos de insatisfação ao fim do dia).")

In [ ]:
# ==============================================================================
# PERGUNTA DE NEGÓCIO 14/14 — Há PADRÕES DE LINGUAGEM que antecipam a
# desistência?
# ==============================================================================
# Expressões (bigramas) enriquecidas nas conversas de risco + posição na
# chamada onde surgem pela primeira vez (0 = início, 1 = fim).
# Leakage: análise descritiva — NÃO entra no modelo.
import pandas as pd
import re
from collections import Counter

cli = df_segmentos[df_segmentos["role"] == "customer"].sort_values("inicio_s")
rot = df_conversas.set_index("id_conversa")["churn_label"]
dur = df_conversas.set_index("id_conversa")["duracao_s"]
textos = cli.groupby("id_conversa")["texto"].apply(" ".join)

def bigramas(t):
    ws = [w for w in re.findall(r"[a-z']+", t.lower()) if len(w) > 2]
    return set(zip(ws, ws[1:]))

bg_churn, bg_nao, n_ch, n_nc = Counter(), Counter(), 0, 0
for idc, t in textos.items():
    if rot[idc] == 1:
        n_ch += 1
        bg_churn.update(bigramas(t))
    else:
        n_nc += 1
        bg_nao.update(bigramas(t))

linhas = []
for (a, b), k in bg_churn.items():
    if k < 8:
        continue
    t_nao = bg_nao.get((a, b), 0)
    tx_c, tx_n = k / n_ch, t_nao / n_nc
    enr = tx_c / max(tx_n, 1 / n_nc)
    if enr >= 5:
        linhas.append({"expressão": f"{a} {b}", "churn": k, "não-churn": t_nao, "enriquecimento": enr})
tab = pd.DataFrame(linhas).sort_values("churn", ascending=False).head(12)
if tab.empty:
    print("Nenhum bigrama cumpre os critérios de enriquecimento (>=5x e >=8 conversas).")
else:
    print(f"Expressões mais características do cliente nas conversas de risco "
          f"({n_ch} risco vs {n_nc} normais):")
    print(tab.round(1).to_string(index=False))

    frase = tab.iloc[0]["expressão"]
    cli_churn = cli[rot.reindex(cli["id_conversa"]).values == 1]
    pos = []
    for idc, g in cli_churn.groupby("id_conversa"):
        m = g[g["texto"].str.lower().str.contains(frase, regex=False)]
        if len(m):
            pos.append(m.iloc[0]["inicio_s"] / dur[idc])
    if pos:
        pos_s = pd.Series(pos)
        print(f"\n'{frase}' surge pela primeira vez, em mediana, a {pos_s.median():.0%} da chamada "
              f"(mín {pos_s.min():.0%}, máx {pos_s.max():.0%}) nas {len(pos)} conversas de risco onde aparece.")
        print(f"\nRESPOSTA: sim — expressões como "
              + ", ".join(f"'{e}'" for e in tab.head(4)["expressão"])
              + f" são típicas das desistências (enriquecimento até {tab['enriquecimento'].max():.0f}x); "
              f"a mais típica ('{frase}') aparece já a {pos_s.median():.0%} da chamada, dando às equipas "
              "tempo de agir antes de o cliente desistir.")
    else:
        print("\nRESPOSTA: existem expressões enriquecidas, mas sem posicionamento resumido.")

In [ ]:
# ==============================================================================
# STEP 10: ENCERRAMENTO — VALIDAÇÃO FINAL E SÍNTESE EXECUTIVA
# ==============================================================================
# Objectivo: fechar o projeto verificando o estado final de todos os artefactos
# (tabelas, label, modelo) e consolidar a síntese executiva com os números
# verificados por execução ao longo das STEP 1–9 e das perguntas P1–P14.
#
# Input:  df_meta, df_conversas, df_segmentos, df_features, df_modelo, modelo
#         final (STEP 8b) e hold-out por falante (X_te, y_te)
# Output: checklist de encerramento + revalidação do modelo + síntese executiva
# ==============================================================================

from sklearn.metrics import (confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)

# --- 1. Checklist de validação final ----------------------------------------------
verificacoes = [
    ("df_meta — metadados das 873 conversas (6 colunas)",
     df_meta.shape == (873, 6)),
    ("df_conversas — 873 conversas, 79 em risco de churn (9,0%)",
     df_conversas.shape[0] == 873 and int(df_conversas["churn_label"].sum()) == 79),
    ("df_segmentos — 94 679 segmentos (11 colunas)",
     df_segmentos.shape == (94679, 11)),
    ("df_features — 873 × 30 features comportamentais/agentes",
     df_features.shape == (873, 30)),
    ("df_modelo — 160 × 54 (79 churn / 81 não-churn)",
     df_modelo.shape == (160, 54) and int(df_modelo["churn_label"].sum()) == 79),
    ("Modelo final treinado (Regressão Logística C=0,03)",
     "final" in globals() and hasattr(globals()["final"], "predict")),
]
print("=== CHECKLIST DE ENCERRAMENTO ===")
tudo_ok = True
for nome, passa in verificacoes:
    tudo_ok = tudo_ok and bool(passa)
    marca = "OK" if passa else "FALHOU"
    print(f"  [{marca}] {nome}")

# --- 2. Revalidação do modelo final (hold-out DISJOINT POR FALANTE, 42 conversas) --
y_f = y_te
pred_f = final.predict(X_te)
proba_f = final.predict_proba(X_te)[:, 1]
f1_f = f1_score(y_f, pred_f)
auc_f = roc_auc_score(y_f, proba_f)
print(f"\nRevalidação do modelo final (42 conversas, 16 falantes nunca vistos): "
      f"F1 {f1_f:.3f} | AUC {auc_f:.3f} | "
      f"precisão {precision_score(y_f, pred_f):.1%} | recall {recall_score(y_f, pred_f):.1%}")
print("Matriz de confusão (linhas=real, colunas=previsto):", confusion_matrix(y_f, pred_f).tolist())

# --- 3. Síntese executiva -------------------------------------------------------
sintese = """
=== SÍNTESE EXECUTIVA (todos os números verificados por execução) ===

DADOS     873 conversas role-played (151,1 h), 16 domínios, 14 acentos,
          94 679 segmentos com diarização — AppTek Callcenter Dialogues.

LABEL     Risco de churn (proxy linguístico): 79/873 (9,0%).
          O TEXTO do cliente nunca é feature — só define o label.

MODELO    Regressão Logística (C=0,03), 51 features (comportamentais,
          acústicas e do agente) — hold-out DISJOINT POR FALANTE:
          F1 0,698 (meta 0,70 falhada por 0,002) | AUC 0,708 |
          precisão 83% quando sinaliza. Evolução 0,552 → 0,698 com
          split por falante + tuning (memorização de vozes eliminada).

USO       Ranker de priorização com limiar p >= 0,50: fila de follow-up
          (18/42 sinalizadas, 83% precisão, 60% recall); top-50% apanha
          68% dos casos — usar como filtro largo, não lista curta.

NEGÓCIO   Motivos: demora (44%) e falha de serviço (42%) > preço.
          Risco por domínio: telecom 22,5%; reclamações: entertainment 48%
          (reclamar ≠ risco de sair). Cliente em risco: interrompe +37%,
          responde mais depressa, voz mais brilhante (ZCR r=+0,19);
          desculpas do agente ACOMPANHAM a fricção (+75%).
          Duração NÃO prevê churn (r=0,02). Sinais antecipatórios
          ('but then', 'help today') já a ~21% da chamada.

LIMITAÇÕES Dataset role-played; label proxy; 160/873 conversas com áudio;
          n pequeno (intervalos largos); sem timestamps (horários impossíveis).

PRÓXIMOS  Processar os 713 WAVs restantes; multi-modal com label externo;
          validar em dados reais de produção.
==================================================================
"""
print(sintese)

estado = "todas as verificações passaram." if tudo_ok else "HÁ VERIFICAÇÕES FALHADAS: rever antes de fechar."
print(f"PROJECTO FECHADO — {estado}")

In [ ]:
# ==============================================================================
# STEP 11: EXPORTAÇÃO PARA PUBLICAÇÃO — ARTEFACTOS KAGGLE
# ==============================================================================
# Objetivo: gerar artefactos de publicação prontos para o Kaggle, com qualidade
# de engenharia de dados: previsões OUT-OF-FOLD (sem optimismo in-sample — cada
# conversa é pontuada por um modelo que nunca viu o falante em treino),
# datasets documentados e README com crédito, metodologia e métricas honestas.
#
# Input:  df_modelo_g (STEP 8b: 160×55 com speaker_cliente), df_conversas (873)
# Output: publicacao_kaggle/ com:
#   - churn_features_oof_160.csv — features + label + churn_score_oof + previsao_risco
#   - conversas_873.csv — nível conversa (agregados + label, sem texto bruto)
#   - README.md — crédito, metodologia, dicionário de dados, métricas
#
# Engenharia: SEED=42 fixo; nº de linhas de cada CSV verificado na leitura;
# anti-leakage mantido (o texto do cliente não está nas features).
# ==============================================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- 1. Previsões out-of-fold (5-fold AGRUPADA por falante) ----------------------
EXCLUIR = ["id_conversa", "churn_label", "ac_dur_real_s"]
CAT_TODAS = ["acento", "dominio", "genero_cliente", "genero_agente"]
df_pub = df_modelo_g.drop(columns=["speaker_cliente"]).copy()
X_todos = df_pub.drop(columns=[c for c in EXCLUIR if c in df_pub.columns])
y_todos = df_pub["churn_label"]
grupos_todos = df_modelo_g["speaker_cliente"].values
num_cols = [c for c in X_todos.columns if c not in CAT_TODAS]

preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_TODAS),
])
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
score_oof = np.full(len(X_todos), np.nan)
for tr, te in sgkf.split(X_todos, y_todos, groups=grupos_todos):
    pipe = Pipeline([("prep", preproc),
                     ("modelo", LogisticRegression(max_iter=3000, C=0.03,
                                                   class_weight="balanced",
                                                   random_state=SEED))])
    pipe.fit(X_todos.iloc[tr], y_todos.iloc[tr])
    score_oof[te] = pipe.predict_proba(X_todos.iloc[te])[:, 1]
assert not np.isnan(score_oof).any(), "há conversas sem score OOF"

df_pub["churn_score_oof"] = score_oof
df_pub["previsao_risco"] = (score_oof >= 0.5).astype(int)
f1_oof = f1_score(y_todos, df_pub["previsao_risco"])
auc_oof = roc_auc_score(y_todos, score_oof)
pr_oof = precision_score(y_todos, df_pub["previsao_risco"])
rc_oof = recall_score(y_todos, df_pub["previsao_risco"])
print(f"Métricas OUT-OF-FOLD (5-fold agrupada por falante, 160 conversas): "
      f"F1 {f1_oof:.3f} | AUC {auc_oof:.3f} | precisão {pr_oof:.1%} | recall {rc_oof:.1%}")

# --- 2. Exportação dos artefactos ------------------------------------------------
PASTA = Path("./publicacao_kaggle")
PASTA.mkdir(parents=True, exist_ok=True)

# 2.1 Dataset de modelação (160) — com score OOF
df_pub.to_csv(PASTA / "churn_features_oof_160.csv", index=False)

# 2.2 Dataset de conversas (873) — agregados + label, sem texto bruto
df_conv_pub = df_conversas.drop(columns=["texto_agente", "texto_cliente"], errors="ignore")
df_conv_pub.to_csv(PASTA / "conversas_873.csv", index=False)

# 2.3 README de publicação
readme = f"""# Previsão de Risco de Churn em Call Center — Áudio (AppTek)

**Autor:** gonaldomichael@gmail.com · **Data:** 2026-11-15 · **SEED:** 42

## Fonte dos dados
AppTek Callcenter Dialogues (Hugging Face: `apptek-com/apptek_callcenter_dialogues`,
config `diarization`), licença CC BY-SA 4.0. 873 conversas role-played em inglês,
151,1 h, 16 domínios, 14 acentos, 94 679 segmentos com diarização, transcrição,
speaker_id e género. Atribuição obrigatória: AppTek / Hugging Face.

## Problema e label
Prever se o cliente indica RISCO DE CHURN a partir de features extraídas do
áudio e dos metadados — NUNCA do texto do cliente (que define o label;
anti-leakage rigoroso). Label proxy (união): churn explícito, frustração
forte, escalonamento, ameaça de saída → 79/873 conversas (9,0%).

## Método
- 51 features: comportamentais (interrupções, latência, ritmo, turnos),
  acústicas do cliente (MFCC, RMS, pitch, ZCR, centróide — librosa) e do
  texto do AGENTE (pedidos de desculpa, empatia).
- Modelo: Regressão Logística (C=0,03, class_weight balanced).
- Avaliação honesta: split e CV AGRUPADOS POR FALANTE (vozes nunca vistas).

## Métricas (verificadas por execução)
- Hold-out disjunto por falante (42 conversas): F1 0,698 | AUC 0,708 |
  precisão 83,3% | recall 60,0% (meta F1 0,70 falhada por 0,002).
- Out-of-fold 5-fold agrupada (as 160 deste dataset): F1 {f1_oof:.3f} |
  AUC {auc_oof:.3f} | precisão {pr_oof:.1%} | recall {rc_oof:.1%}.
- Uso recomendado: ranker de priorização (limiar 0,50; top-50% apanha 68%
  dos casos com 81% de precisão).

## Ficheiros
- `churn_features_oof_160.csv` — 160 conversas com features, `churn_label`,
  `churn_score_oof` (previsão out-of-fold) e `previsao_risco` (limiar 0,50).
- `conversas_873.csv` — as 873 conversas com agregados e `churn_label`.

## Dicionário (grupos de colunas)
- `id_conversa`, `acento`, `dominio`, `genero_*`: metadados.
- `duracao_s`, `n_segmentos`, `dur_fala_*`, `turnos_*`, `interrupcoes_*`,
  `latencia_*`, `pct_silencio`, `ritmo_*`, `n_palavras_*`: comportamentais.
- `ag_*`: padrões do texto do agente (desculpas, empatia, agradecimentos).
- `ac_*`: acústicas (MFCC1–13, RMS, pitch, ZCR, centróide; `ac_cli_*` = só
  nos segmentos do cliente).
- `churn_label`: alvo. `churn_score_oof`/`previsao_risco`: previsões OOF.

## Limitações
Conversas role-played (não produção); label proxy derivado do texto; áudio
processado numa amostra estratificada de 160 conversas; n pequeno →
intervalos largos. Detalhes completos no notebook original.
"""
(PASTA / "README.md").write_text(readme, encoding="utf-8")

# --- 3. Manifesto de artefactos (verificação por releitura) ----------------------
print("\n=== Manifesto de artefactos (linhas verificadas por releitura) ===")
esperados = {"churn_features_oof_160.csv": 160, "conversas_873.csv": 873}
tudo_ok = True
for f in sorted(PASTA.iterdir()):
    if f.suffix == ".csv":
        n = len(pd.read_csv(f))
        ok = esperados.get(f.name) == n
        tudo_ok &= ok
        marca = "OK" if ok else "ERRADO"
        print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB  linhas={n}  [{marca}]")
    else:
        print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")

print(f"\nSTEP 11 concluído: artefactos Kaggle exportados para {PASTA} — "
      + ("todos os ficheiros com o nº de linhas esperado." if tudo_ok else "VERIFICAR linhas!"))

In [ ]:
# ==============================================================================
# STEP 12: VISUALIZAÇÕES — TENDÊNCIAS E MOTIVOS DE CONTACTO (VISÃO DA GESTÃO)
# ==============================================================================
# Objetivo: 3 gráficos que ajudam a gestão a perceber o que gera volume e risco:
#   1. NUVEM DE PALAVRAS — termos mais mencionados pelos clientes (873)
#   2. LINHAS MÚLTIPLAS (TENDÊNCIAS) — evolução dos temas de risco. NOTA
#      HONESTA: o dataset NÃO tem datas (P13) — impossível evolução
#      diária/mensal; a tendência é ao longo da LINHA TEMPORAL DA CHAMADA
#      (decis da duração): quando é que cada tema surge na conversa.
#   3. BARRAS HORIZONTAIS (ALERTAS DE RISCO) — componentes de alerta e
#      motivos nas 79 conversas de risco (padrões exatos do STEP 5 e do P8).
#
# Input:  df_conversas (873), df_segmentos (94.679)
# Output: 3 gráficos renderizados + 3 PNGs em publicacao_kaggle/
#
# Anti-leakage: gráficos DESCRITIVOS de negócio (como P8/P9) — o texto do
# cliente aqui não alimenta o modelo, apenas descreve o que se passa.
#
# Nota de ambiente: a célula instala o 'wordcloud' programaticamente se
# faltar (auto-suficiente para Run All — não depende de célula %pip à parte).
# ==============================================================================

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

try:
    from wordcloud import WordCloud
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wordcloud"], check=True)
    from wordcloud import WordCloud

PASTA = Path("./publicacao_kaggle")
PASTA.mkdir(parents=True, exist_ok=True)

# --- Gráfico 1: NUVEM DE PALAVRAS — o que os clientes mais mencionam -------------
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "of", "to", "in", "on", "at", "for",
    "with", "about", "as", "is", "are", "was", "were", "be", "been", "am", "do", "does",
    "did", "have", "has", "had", "will", "would", "can", "could", "should", "may",
    "might", "must", "i", "im", "ive", "me", "my", "mine", "you", "your", "yours", "u",
    "he", "she", "it", "its", "we", "us", "our", "ours", "they", "them", "their", "this",
    "that", "these", "those", "there", "here", "what", "when", "where", "which", "who", "why",
    "how", "not", "no", "yes", "so", "just", "also", "very", "really", "too", "then",
    "than", "now", "get", "got", "gonna", "going", "want", "wanted", "need", "needed",
    "know", "knew", "like", "okay", "ok", "oh", "um", "uh", "well", "right", "sure",
    "sorry", "please", "thank", "thanks", "yeah", "yep", "hello", "hi", "hey", "goodbye",
    "bye", "see", "say", "said", "says", "tell", "told", "let", "lets", "dont", "cant",
    "wont", "isnt", "didnt", "doesnt", "thats", "one", "something", "anything",
    "everything", "nothing", "thing", "things", "still", "back", "much", "many", "more",
    "some", "any", "all", "because", "from", "by", "up", "down", "out", "over", "under",
    "again", "ever", "never", "always", "like",
    # fillers conversacionais detetados no top-10 da 1.ª execução
    "i'm", "i'll", "i'd", "i've", "it's", "that's", "there's", "don't",
    "didn't", "can't", "won't", "ohh", "ohhh", "ah", "hmm", "think",
    "two", "three", "good", "great", "go", "make", "take", "come",
    "look", "ask", "put", "give", "actually", "maybe", "kind", "sort",
    # números ditos e acknowledgments (ruído para a gestão — 2.ª execução)
    "five", "four", "seven", "six", "eight", "nine", "ten", "twenty",
    "hundred", "mhm", "mm", "perfect", "wonderful", "awesome",
    "fantastic", "excellent", "absolutely",
}
todo_texto = " ".join(df_conversas["texto_cliente"].astype(str)).lower()
wc = WordCloud(width=1400, height=800, background_color="white", colormap="cividis",
               stopwords=STOPWORDS, max_words=80, collocations=False,
               prefer_horizontal=0.95, random_state=SEED).generate(todo_texto)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("O que os clientes mais mencionam — nuvem de palavras (873 conversas, texto do cliente)", fontsize=13)
fig.tight_layout()
fig.savefig(PASTA / "grafico_1_nuvem_palavras_cliente.png", dpi=150)
plt.show()
top10 = sorted(wc.words_.items(), key=lambda kv: -kv[1])[:10]
print("Top-10 termos (peso relativo):", ", ".join(f"{w} ({v:.2f})" for w, v in top10))

# --- Gráfico 2: TENDÊNCIAS — evolução dos temas ao longo da CHAMADA ------------
# Sem datas no dataset (P13): a evolução é intra-chamada, em decis da duração.
OBJ = r"(?:account|service|subscription|plan|policy|contract|membership)"
JAN = r"[\w\s,'()-]{0,40}"
TEMAS = {
    "Churn explícito (cancelar/fechar)":
        rf"\b(?:cancel|close|terminat)\w*\b{JAN}\b{OBJ}\b",
    "Frustração forte":
        r"\b(?:fed\s+up|sick\s+and\s+tired|sick\s+of|ridiculous|unacceptable|"
        r"worst|terrible|awful|disgusting|disgusted|appalled|furious|"
        r"frustrated|frustrating|disappointed|disappointing|unhappy|not\s+happy|"
        r"dissatisfied|poor\s+service|bad\s+service|what\s+a\s+joke)\b",
    "Escalonamento (gestor/queixa)":
        r"\b(?:speak\s+(?:to|with)\s+(?:a\s+|the\s+)?(?:manager|supervisor)|"
        r"talk\s+to\s+(?:a\s+|the\s+)?(?:manager|supervisor)|"
        r"your\s+(?:manager|supervisor)|"
        r"file\s+(?:a\s+)?complaint|make\s+(?:a\s+)?complaint|"
        r"report\s+(?:this|that|it)|take\s+this\s+further|"
        r"ombudsman|legal\s+action|lawyer|sue\s+you)\b",
    "Ameaça de saída":
        r"\b(?:take\s+my\s+business\s+elsewhere|"
        r"go(?:ing)?\s+(?:to|over\s+to)\s+(?:another|a\s+different)\s+"
        r"(?:provider|company|bank|airline|carrier|supplier|competitor)|"
        r"moving\s+to\s+(?:another|a\s+different)\s+(?:provider|company|bank|airline|carrier)|"
        r"switch(?:ing)?\s+(?:banks?|providers?|carriers?|companies)|"
        r"never\s+(?:using|coming)\s+(?:you\s+)?again|"
        r"done\s+with\s+(?:you|this))\b",
}

cli = df_segmentos[df_segmentos["role"] == "customer"].copy()
dur_conv = df_conversas.set_index("id_conversa")["duracao_s"]
cli["duracao_s"] = cli["id_conversa"].map(dur_conv)
cli["decil"] = ((cli["inicio_s"] + cli["fim_s"]) / 2 / cli["duracao_s"] * 10).astype(int).clip(0, 9)
n_conv = len(df_conversas)

fig, ax = plt.subplots(figsize=(10.5, 6))
cores_t = {"Churn explícito (cancelar/fechar)": "#c0392b", "Frustração forte": "#e67e22",
           "Escalonamento (gestor/queixa)": "#8e44ad", "Ameaça de saída": "#2980b9"}
for nome, padrao in TEMAS.items():
    m = cli["texto"].str.contains(padrao, case=False, regex=True, na=False)
    evo = (cli[m].groupby("decil")["id_conversa"].nunique() / n_conv * 100)
    serie = [evo.get(d, 0.0) for d in range(10)]
    ax.plot(range(1, 11), serie, marker="o", lw=2.2, color=cores_t[nome], label=nome)
ax.set_xticks(range(1, 11))
ax.set_xticklabels([f"{d*10}–{d*10+10}%" for d in range(10)], fontsize=8)
ax.set_xlabel("Posição na chamada (decis da duração — sem datas no dataset, P13)")
ax.set_ylabel("% das 873 conversas com o tema")
ax.set_title("Tendência dos temas de risco ao longo da chamada\nquando surge cada tema — sinal antecipado é oportunidade de retenção")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PASTA / "grafico_2_evolucao_temas_chamada.png", dpi=150)
plt.show()

# --- Gráfico 3: ALERTAS DE RISCO — barras horizontais ----------------------------
# Painel esquerdo: componentes do alerta (gravidade, padrões do STEP 5);
# painel direito: motivos mencionados (padrões EXATOS do P8 — números iguais à tabela).
risco = df_conversas[df_conversas["churn_label"] == 1]
texto_risco = risco["texto_cliente"].astype(str).str.lower()
comp = {
    "Churn explícito (A)": int(texto_risco.str.contains(TEMAS["Churn explícito (cancelar/fechar)"], regex=True).sum()),
    "Frustração forte (B)": int(texto_risco.str.contains(TEMAS["Frustração forte"], regex=True).sum()),
    "Escalonamento (C)": int(texto_risco.str.contains(TEMAS["Escalonamento (gestor/queixa)"], regex=True).sum()),
    "Ameaça de saída (D)": int(texto_risco.str.contains(TEMAS["Ameaça de saída"], regex=True).sum()),
}
MOTIVOS = {
    "preço/custos":        r"\b(price|fee|charge|expensive|cost|billing|payment)\b",
    "problema de serviço": r"\b(broken|faulty|defect|not work|doesn't work|stopped working|error|issue|problem)\b",
    "demora/atraso":       r"\b(wait|delay|late|waiting|slow)\b",
    "mudança de operador": r"\b(switch|another provider|competitor|elsewhere|other company)\b",
    "atendimento ruim":    r"\b(rude|unhelpful|useless|incompetent|supervisor|speak to a manager)\b",
    "reembolso/dinheiro":  r"\b(refund|reimburse|money back|compensation)\b",
}
mot = pd.Series({m: int(texto_risco.str.contains(rx, regex=True).sum()) for m, rx in MOTIVOS.items()}).sort_values()
comp_s = pd.Series(comp).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.5))
ax1.barh(list(comp_s.index), comp_s.values, color="#c0392b")
for i, v in enumerate(comp_s.values):
    ax1.text(v + 0.4, i, f"{v} ({v/len(risco):.0%})", va="center", fontsize=9)
ax1.set_xlabel(f"Conversas de risco (n={len(risco)}, com sobreposição)")
ax1.set_title("Componentes do alerta (gravidade)")
ax1.set_xlim(0, comp_s.max() * 1.28)
ax1.spines[["top", "right"]].set_visible(False)

ax2.barh(list(mot.index), mot.values, color="#7fb3d5")
for i, v in enumerate(mot.values):
    ax2.text(v + 0.4, i, f"{v} ({v/len(risco):.0%})", va="center", fontsize=9)
ax2.set_xlabel(f"Conversas de risco (n={len(risco)}, com sobreposição)")
ax2.set_title("Motivos mencionados (P8)")
ax2.set_xlim(0, mot.max() * 1.28)
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle("Alertas de risco — o que dispara o sinal e porquê (79 conversas)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PASTA / "grafico_3_alertas_risco_motivos.png", dpi=150)
plt.show()

# --- Verificação ----------------------------------------------------------------
print("=== Verificação de consistência (painel direito vs P8) ===")
esperado_p8 = {"demora/atraso": 35, "problema de serviço": 33, "preço/custos": 32,
               "reembolso/dinheiro": 17, "atendimento ruim": 6, "mudança de operador": 5}
tudo_ok = all(mot[k] == v for k, v in esperado_p8.items())
for k, v in esperado_p8.items():
    marca = "OK" if mot[k] == v else "DIFERENTE"
    print(f"  {k:22s} P8={v:3d}  gráfico={int(mot[k]):3d}  [{marca}]")

print("\n=== PNGs exportados ===")
for f in sorted(PASTA.glob("*.png")):
    print(f"  {f.name:44s} {f.stat().st_size / 1024:8.1f} KB")
print("\nSTEP 12 concluído: nuvem de palavras, tendências e alertas de risco renderizados.")